# FPL Season Retrospective

This notebook will analyse FPL manager team ID `816200`, compare the season against a representative top-N sample, and generate next-season decision rules.

Codex should build this notebook one story at a time using `AGENTS.md`, `BACKLOG.md`, `KANBAN.md`, `SPRINTS.md`, `STATUS.md`, and `TESTING.md`.


## 01 Config and Project Setup

Story 1.1 should populate this section.


In [25]:
from pathlib import Path

# Core analysis config
MY_TEAM_ID = 816200
TOP_N = 80_000
SAMPLE_SIZE = 1000
SEASON_GWS = list(range(1, 39))
RANDOM_SEED = 42
CACHE_ENABLED = True
REQUEST_SLEEP_SECONDS = 0.25
MAX_MANAGERS_FOR_PICKS = 1000

# Project paths
NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR

DATA_DIR = PROJECT_ROOT / 'data'
DATA_RAW_DIR = DATA_DIR / 'raw'
DATA_PROCESSED_DIR = DATA_DIR / 'processed'
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'
OUTPUT_TABLES_DIR = OUTPUTS_DIR / 'tables'
OUTPUT_CHARTS_DIR = OUTPUTS_DIR / 'charts'

REQUIRED_FOLDERS = {
    'data_raw': DATA_RAW_DIR,
    'data_processed': DATA_PROCESSED_DIR,
    'output_tables': OUTPUT_TABLES_DIR,
    'output_charts': OUTPUT_CHARTS_DIR,
}

for path in REQUIRED_FOLDERS.values():
    path.mkdir(parents=True, exist_ok=True)

print('FPL retrospective config')
print(f'Project root: {PROJECT_ROOT}')
print(f'Manager team ID: {MY_TEAM_ID}')
print(f'Top-N threshold: {TOP_N:,}')
print(f'Sample size: {SAMPLE_SIZE:,}')
print(f'Season gameweeks: {min(SEASON_GWS)}-{max(SEASON_GWS)} ({len(SEASON_GWS)} total)')
print(f'Random seed: {RANDOM_SEED}')
print(f'Cache enabled: {CACHE_ENABLED}')
print(f'Request sleep seconds: {REQUEST_SLEEP_SECONDS}')
print(f'Max managers for picks: {MAX_MANAGERS_FOR_PICKS:,}')
print('Required folders:')
for name, path in REQUIRED_FOLDERS.items():
    print(f'  - {name}: {path} [{"ok" if path.exists() else "missing"}]')

FPL retrospective config
Project root: /Users/djay/Documents/fpl-retrospective-codex-starter
Manager team ID: 816200
Top-N threshold: 80,000
Sample size: 1,000
Season gameweeks: 1-38 (38 total)
Random seed: 42
Cache enabled: True
Request sleep seconds: 0.25
Max managers for picks: 1,000
Required folders:
  - data_raw: /Users/djay/Documents/fpl-retrospective-codex-starter/data/raw [ok]
  - data_processed: /Users/djay/Documents/fpl-retrospective-codex-starter/data/processed [ok]
  - output_tables: /Users/djay/Documents/fpl-retrospective-codex-starter/outputs/tables [ok]
  - output_charts: /Users/djay/Documents/fpl-retrospective-codex-starter/outputs/charts [ok]


## 02 Decision Quality Framework

This retrospective separates decision process from decision outcome. Process quality is judged using only information that would have been available before the relevant gameweek or deadline. Outcome quality is judged after the fact from the points gained or lost.

That split matters because a strong FPL decision can fail through variance, injuries, rotation, or low-probability match events. A weak decision can also succeed by luck. The analysis should preserve that distinction instead of rewriting the decision after seeing the result.

**Hindsight leakage warning:** do not use future points, final season totals, future ownership, future rank, or any other post-decision information when judging whether the original process was good.


In [15]:
DECISION_QUALITY_LABELS = {
    'good_process_good_outcome': 'Good process, good outcome',
    'good_process_bad_outcome': 'Good process, bad outcome',
    'bad_process_good_outcome': 'Bad process, good outcome',
    'bad_process_bad_outcome': 'Bad process, bad outcome',
}

assert len(DECISION_QUALITY_LABELS) == 4
assert all(isinstance(label, str) and len(label.split()) >= 4 for label in DECISION_QUALITY_LABELS.values())

print('Decision quality labels')
for key, label in DECISION_QUALITY_LABELS.items():
    print(f'- {key}: {label}')

Decision quality labels
- good_process_good_outcome: Good process, good outcome
- good_process_bad_outcome: Good process, bad outcome
- bad_process_good_outcome: Bad process, good outcome
- bad_process_bad_outcome: Bad process, bad outcome


## 03 API Fetch and Cache Helpers

Story 2.1 introduces reusable helpers for fetching FPL API JSON and caching raw responses locally. Later notebook sections should call these helpers rather than using `requests` directly.


In [16]:
import sys

SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from fpl_retro.api import fetch_json
from fpl_retro.cache import load_json

BOOTSTRAP_STATIC_URL = 'https://fantasy.premierleague.com/api/bootstrap-static/'
BOOTSTRAP_SMOKE_CACHE = DATA_RAW_DIR / 'bootstrap_static_smoke.json'

bootstrap_smoke = fetch_json(
    BOOTSTRAP_STATIC_URL,
    cache_path=BOOTSTRAP_SMOKE_CACHE,
    force_refresh=not CACHE_ENABLED,
)
bootstrap_cached = fetch_json(BOOTSTRAP_STATIC_URL, cache_path=BOOTSTRAP_SMOKE_CACHE)
bootstrap_reloaded = load_json(BOOTSTRAP_SMOKE_CACHE)

expected_bootstrap_keys = {'events', 'elements', 'teams', 'element_types'}
missing_bootstrap_keys = expected_bootstrap_keys - set(bootstrap_smoke)

assert BOOTSTRAP_SMOKE_CACHE.exists()
assert not missing_bootstrap_keys
assert bootstrap_cached == bootstrap_reloaded

print('Bootstrap smoke test passed')
print(f'Cache path: {BOOTSTRAP_SMOKE_CACHE}')
print(f'Cached file exists: {BOOTSTRAP_SMOKE_CACHE.exists()}')
print(f'Expected keys present: {sorted(expected_bootstrap_keys)}')


Bootstrap smoke test passed
Cache path: /Users/djay/Documents/fpl-retrospective-codex-starter/data/raw/bootstrap_static_smoke.json
Cached file exists: True
Expected keys present: ['element_types', 'elements', 'events', 'teams']


## 04 Bootstrap Static Metadata

This section loads `bootstrap-static` through the cached API helper and creates the core metadata tables used by later joins.


In [17]:
import pandas as pd

BOOTSTRAP_CACHE = DATA_RAW_DIR / 'bootstrap_static_smoke.json'
bootstrap_static = fetch_json(
    BOOTSTRAP_STATIC_URL,
    cache_path=BOOTSTRAP_CACHE,
    force_refresh=not CACHE_ENABLED,
)

expected_bootstrap_keys = {'events', 'elements', 'teams', 'element_types'}
missing_bootstrap_keys = expected_bootstrap_keys - set(bootstrap_static)
assert not missing_bootstrap_keys, f'Missing bootstrap keys: {sorted(missing_bootstrap_keys)}'

players_raw_df = pd.DataFrame(bootstrap_static['elements'])
teams_df = pd.DataFrame(bootstrap_static['teams'])
events_df = pd.DataFrame(bootstrap_static['events'])
element_types_df = pd.DataFrame(bootstrap_static['element_types'])

team_lookup_df = teams_df[['id', 'name', 'short_name']].rename(
    columns={'id': 'team', 'name': 'team_name', 'short_name': 'team_short_name'}
)
position_lookup_df = element_types_df[
    ['id', 'singular_name', 'singular_name_short']
].rename(
    columns={
        'id': 'element_type',
        'singular_name': 'position',
        'singular_name_short': 'position_short',
    }
)

players_enriched_df = players_raw_df.merge(team_lookup_df, on='team', how='left').merge(
    position_lookup_df, on='element_type', how='left'
)
players_enriched_df['price'] = players_enriched_df['now_cost'] / 10

preferred_player_columns = [
    'id',
    'web_name',
    'first_name',
    'second_name',
    'team',
    'team_name',
    'team_short_name',
    'element_type',
    'position',
    'position_short',
    'price',
    'now_cost',
    'selected_by_percent',
    'form',
    'total_points',
    'minutes',
    'points_per_game',
    'expected_goals',
    'expected_assists',
    'expected_goal_involvements',
    'expected_goals_conceded',
]
players_df = players_enriched_df[
    [column for column in preferred_player_columns if column in players_enriched_df.columns]
].copy()

players_df.to_csv(DATA_PROCESSED_DIR / 'players.csv', index=False)
teams_df.to_csv(DATA_PROCESSED_DIR / 'teams.csv', index=False)
events_df.to_csv(DATA_PROCESSED_DIR / 'events.csv', index=False)
element_types_df.to_csv(DATA_PROCESSED_DIR / 'element_types.csv', index=False)

assert len(players_df) > 0
assert len(teams_df) == 20
assert len(element_types_df) == 4
assert players_df['team_name'].notna().all()
assert players_df['position'].notna().all()
assert players_df['id'].is_unique

required_output_files = [
    DATA_PROCESSED_DIR / 'players.csv',
    DATA_PROCESSED_DIR / 'teams.csv',
    DATA_PROCESSED_DIR / 'events.csv',
    DATA_PROCESSED_DIR / 'element_types.csv',
]
assert all(path.exists() for path in required_output_files)

print('Bootstrap metadata processed')
print(f'players_df shape: {players_df.shape}')
print(f'teams_df shape: {teams_df.shape}')
print(f'events_df shape: {events_df.shape}')
print(f'element_types_df shape: {element_types_df.shape}')
print('Player sample:')
print(players_df.head().to_string(index=False))


Bootstrap metadata processed
players_df shape: (841, 21)
teams_df shape: (20, 22)
events_df shape: (38, 29)
element_types_df shape: (4, 13)
Player sample:
 id     web_name first_name           second_name  team team_name team_short_name  element_type   position position_short  price  now_cost selected_by_percent form  total_points  minutes points_per_game expected_goals expected_assists expected_goal_involvements expected_goals_conceded
  1         Raya      David           Raya Martín     1   Arsenal             ARS             1 Goalkeeper            GKP    6.2        62                36.4  5.2           162     3330             4.4           0.00             0.07                       0.07                   27.56
  2 Arrizabalaga       Kepa Arrizabalaga Revuelta     1   Arsenal             ARS             1 Goalkeeper            GKP    4.0        40                 0.4  0.5             2       90             2.0           0.00             0.00                       0.00            

## 05 Fixtures

This section loads the FPL fixtures endpoint, joins home and away team names, and saves the processed fixture table for later fixture difficulty and team strength work.


In [18]:
FIXTURES_URL = 'https://fantasy.premierleague.com/api/fixtures/'
FIXTURES_CACHE = DATA_RAW_DIR / 'fixtures.json'

fixtures_json = fetch_json(
    FIXTURES_URL,
    cache_path=FIXTURES_CACHE,
    force_refresh=not CACHE_ENABLED,
)
fixtures_df = pd.DataFrame(fixtures_json)

home_team_lookup_df = teams_df[['id', 'name', 'short_name']].rename(
    columns={
        'id': 'team_h',
        'name': 'team_h_name',
        'short_name': 'team_h_short_name',
    }
)
away_team_lookup_df = teams_df[['id', 'name', 'short_name']].rename(
    columns={
        'id': 'team_a',
        'name': 'team_a_name',
        'short_name': 'team_a_short_name',
    }
)

fixtures_df = fixtures_df.merge(home_team_lookup_df, on='team_h', how='left').merge(
    away_team_lookup_df, on='team_a', how='left'
)

fixtures_output_path = DATA_PROCESSED_DIR / 'fixtures.csv'
fixtures_df.to_csv(fixtures_output_path, index=False)

required_fixture_columns = {'id', 'team_h', 'team_a', 'team_h_name', 'team_a_name'}
missing_fixture_columns = required_fixture_columns - set(fixtures_df.columns)

assert len(fixtures_df) > 0
assert not missing_fixture_columns, f'Missing fixture columns: {sorted(missing_fixture_columns)}'
assert fixtures_df['team_h_name'].notna().all()
assert fixtures_df['team_a_name'].notna().all()
assert fixtures_df['id'].is_unique
assert fixtures_output_path.exists()

missing_event_count = fixtures_df['event'].isna().sum() if 'event' in fixtures_df.columns else None

print('Fixtures processed')
print(f'fixtures_df shape: {fixtures_df.shape}')
print(f'Missing event count: {missing_event_count}')
print(f'Output path: {fixtures_output_path}')
print('Fixture sample:')
print(fixtures_df[['id', 'event', 'team_h_name', 'team_a_name']].head().to_string(index=False))


Fixtures processed
fixtures_df shape: (380, 21)
Missing event count: 0
Output path: /Users/djay/Documents/fpl-retrospective-codex-starter/data/processed/fixtures.csv
Fixture sample:
 id  event team_h_name team_a_name
  1      1   Liverpool Bournemouth
  2      1 Aston Villa   Newcastle
  3      1    Brighton      Fulham
  6      1       Spurs     Burnley
  5      1  Sunderland    West Ham


## 06 Gameweek Live Player Data

This section loads every `event/{gw}/live` endpoint, flattens player match stats into one player-gameweek table, and preserves FPL scoring-driver fields for later decision analysis.


In [19]:
import time

GW_LIVE_RAW_DIR = DATA_RAW_DIR / 'gw_live'
GW_LIVE_RAW_DIR.mkdir(parents=True, exist_ok=True)

gw_live_rows = []
for gw in SEASON_GWS:
    gw_live_url = f'https://fantasy.premierleague.com/api/event/{gw}/live/'
    gw_live_cache = GW_LIVE_RAW_DIR / f'gw_{gw:02d}_live.json'
    cache_was_missing = not gw_live_cache.exists()

    gw_live_json = fetch_json(
        gw_live_url,
        cache_path=gw_live_cache,
        force_refresh=not CACHE_ENABLED,
    )

    for element in gw_live_json.get('elements', []):
        row = {
            'gameweek': gw,
            'player_id': element.get('id'),
        }
        row.update(element.get('stats', {}))
        gw_live_rows.append(row)

    if cache_was_missing and REQUEST_SLEEP_SECONDS:
        time.sleep(REQUEST_SLEEP_SECONDS)

gw_live_stats_df = pd.DataFrame(gw_live_rows)

player_metadata_columns = [
    'id',
    'web_name',
    'team_name',
    'team_short_name',
    'position',
    'position_short',
    'price',
]
player_lookup_for_live_df = players_df[
    [column for column in player_metadata_columns if column in players_df.columns]
].rename(columns={'id': 'player_id'})

gw_live_df = gw_live_stats_df.merge(player_lookup_for_live_df, on='player_id', how='left')

scoring_driver_columns = [
    'gameweek',
    'player_id',
    'web_name',
    'team_name',
    'position',
    'position_short',
    'price',
    'total_points',
    'minutes',
    'starts',
    'goals_scored',
    'assists',
    'clean_sheets',
    'goals_conceded',
    'saves',
    'penalties_saved',
    'penalties_missed',
    'yellow_cards',
    'red_cards',
    'own_goals',
    'bonus',
    'bps',
    'defensive_contribution',
    'clearances_blocks_interceptions',
    'recoveries',
    'expected_goals',
    'expected_assists',
    'expected_goal_involvements',
    'expected_goals_conceded',
]
front_columns = [column for column in scoring_driver_columns if column in gw_live_df.columns]
remaining_columns = [column for column in gw_live_df.columns if column not in front_columns]
gw_live_df = gw_live_df[front_columns + remaining_columns].copy()

gw_live_output_path = DATA_PROCESSED_DIR / 'gw_live.csv'
gw_live_df.to_csv(gw_live_output_path, index=False)

required_live_columns = {
    'gameweek',
    'player_id',
    'web_name',
    'team_name',
    'position',
    'total_points',
    'minutes',
    'goals_scored',
    'assists',
    'clean_sheets',
    'bonus',
    'bps',
}
missing_live_columns = required_live_columns - set(gw_live_df.columns)
expected_metric_columns = {
    'expected_goals',
    'expected_assists',
    'expected_goal_involvements',
    'expected_goals_conceded',
}
missing_expected_metric_columns = expected_metric_columns - set(gw_live_df.columns)
expected_gameweeks = set(SEASON_GWS)
actual_gameweeks = set(gw_live_df['gameweek'].dropna().astype(int))

assert not missing_live_columns, f'Missing live columns: {sorted(missing_live_columns)}'
assert not missing_expected_metric_columns, f'Missing expected metric columns: {sorted(missing_expected_metric_columns)}'
assert actual_gameweeks == expected_gameweeks
assert len(gw_live_df) >= len(SEASON_GWS) * 500
assert gw_live_df['web_name'].notna().all()
assert gw_live_df['team_name'].notna().all()
assert gw_live_df[['gameweek', 'player_id']].duplicated().sum() == 0
assert gw_live_output_path.exists()

print('Gameweek live player data processed')
print(f'gw_live_df shape: {gw_live_df.shape}')
print(f'Gameweeks present: {min(actual_gameweeks)}-{max(actual_gameweeks)} ({len(actual_gameweeks)} total)')
print(f'Raw cache files: {len(list(GW_LIVE_RAW_DIR.glob("gw_*_live.json")))}')
print(f'Output path: {gw_live_output_path}')
print('Live data sample:')
sample_columns = [column for column in scoring_driver_columns if column in gw_live_df.columns][:18]
print(gw_live_df[sample_columns].head().to_string(index=False))


Gameweek live player data processed
gw_live_df shape: (29747, 37)
Gameweeks present: 1-38 (38 total)
Raw cache files: 38
Output path: /Users/djay/Documents/fpl-retrospective-codex-starter/data/processed/gw_live.csv
Live data sample:
 gameweek  player_id     web_name team_name   position position_short  price  total_points  minutes  starts  goals_scored  assists  clean_sheets  goals_conceded  saves  penalties_saved  penalties_missed  yellow_cards
        1          1         Raya   Arsenal Goalkeeper            GKP    6.2            10       90       1             0        0             1               0      7                0                 0             1
        1          2 Arrizabalaga   Arsenal Goalkeeper            GKP    4.0             0        0       0             0        0             0               0      0                0                 0             0
        1          3         Hein   Arsenal Goalkeeper            GKP    4.0             0        0       0         

## 07 Overall League Standings

This section fetches Overall league standings pages so later stories can build a representative top-N manager sample. It defaults to a small smoke fetch and can be switched to full-fetch mode after review.


In [20]:
OVERALL_LEAGUE_ID = 314
STANDINGS_PAGE_SIZE = 50
STANDINGS_PHASE = 1
STANDINGS_FULL_FETCH = False
STANDINGS_SMOKE_PAGES = [1, 2]
STANDINGS_PAGES_NEEDED = (TOP_N + STANDINGS_PAGE_SIZE - 1) // STANDINGS_PAGE_SIZE

STANDINGS_RAW_DIR = DATA_RAW_DIR / 'standings'
STANDINGS_RAW_DIR.mkdir(parents=True, exist_ok=True)

def standings_page_url(page_num: int) -> str:
    return (
        f'https://fantasy.premierleague.com/api/leagues-classic/{OVERALL_LEAGUE_ID}/'
        f'standings/?page_standings={page_num}&phase={STANDINGS_PHASE}'
    )

standings_pages_to_fetch = (
    list(range(1, STANDINGS_PAGES_NEEDED + 1))
    if STANDINGS_FULL_FETCH
    else STANDINGS_SMOKE_PAGES
)

standings_rows = []
for page_num in standings_pages_to_fetch:
    standings_cache = STANDINGS_RAW_DIR / f'overall_page_{page_num:04d}.json'
    cache_was_missing = not standings_cache.exists()
    standings_json = fetch_json(
        standings_page_url(page_num),
        cache_path=standings_cache,
        force_refresh=not CACHE_ENABLED,
    )
    results = standings_json.get('standings', {}).get('results', [])

    for result in results:
        row = dict(result)
        row['page_standings'] = page_num
        row['manager_id'] = result.get('entry')
        row['overall_rank'] = result.get('rank')
        standings_rows.append(row)

    if cache_was_missing and REQUEST_SLEEP_SECONDS:
        time.sleep(REQUEST_SLEEP_SECONDS)

top_n_standings_smoke_df = pd.DataFrame(standings_rows)
if not top_n_standings_smoke_df.empty:
    top_n_standings_smoke_df = top_n_standings_smoke_df[
        top_n_standings_smoke_df['overall_rank'].astype(int) <= TOP_N
    ].copy()

standings_output_path = DATA_PROCESSED_DIR / 'top_n_standings_smoke.csv'
top_n_standings_smoke_df.to_csv(standings_output_path, index=False)

required_standings_columns = {'manager_id', 'overall_rank', 'page_standings'}
missing_standings_columns = required_standings_columns - set(top_n_standings_smoke_df.columns)

assert STANDINGS_PAGES_NEEDED == ((TOP_N + STANDINGS_PAGE_SIZE - 1) // STANDINGS_PAGE_SIZE)
assert len(standings_pages_to_fetch) >= 1
assert not top_n_standings_smoke_df.empty
assert not missing_standings_columns, f'Missing standings columns: {sorted(missing_standings_columns)}'
assert top_n_standings_smoke_df['manager_id'].notna().all()
assert top_n_standings_smoke_df['overall_rank'].notna().all()
assert set(top_n_standings_smoke_df['page_standings']) == set(standings_pages_to_fetch)
assert standings_output_path.exists()

print('Overall standings smoke fetch complete')
print(f'Overall league ID: {OVERALL_LEAGUE_ID}')
print(f'Top-N threshold: {TOP_N:,}')
print(f'Pages needed for full fetch: {STANDINGS_PAGES_NEEDED:,}')
print(f'Pages fetched this run: {standings_pages_to_fetch}')
print(f'top_n_standings_smoke_df shape: {top_n_standings_smoke_df.shape}')
print(f'Output path: {standings_output_path}')
print('Standings sample:')
display_columns = [
    column for column in ['manager_id', 'overall_rank', 'entry_name', 'player_name', 'total', 'page_standings']
    if column in top_n_standings_smoke_df.columns
]
print(top_n_standings_smoke_df[display_columns].head().to_string(index=False))


Overall standings smoke fetch complete
Overall league ID: 314
Top-N threshold: 80,000
Pages needed for full fetch: 1,600
Pages fetched this run: [1, 2]
top_n_standings_smoke_df shape: (100, 14)
Output path: /Users/djay/Documents/fpl-retrospective-codex-starter/data/processed/top_n_standings_smoke.csv
Standings sample:
 manager_id  overall_rank           entry_name    player_name  total  page_standings
    3027768             1                Ibsen     Erik Ibsen   2582               1
    8142824             2 The Honest Trundlers     Ian Foster   2544               1
    3244539             3            Team Peki     Ivan Peula   2538               1
    5093997             4           กรุ้มกริ่ม        T-T T-T   2535               1
     785300             5  Soucek Yourself Son David whitlock   2533               1


## 08 Rank-Stratified Top-N Sample

This section creates a reproducible comparison-manager sample from the parsed Overall standings candidate file. Manager `816200` is excluded from the comparison sample and remains the focal manager.


In [21]:
from fpl_retro.sampling import DEFAULT_TOP_80K_RANK_BANDS, create_rank_stratified_sample

standings_candidates_path = DATA_PROCESSED_DIR / 'top_n_standings_candidates.csv'
MIN_SAMPLE_MANAGERS_PER_RANK_BAND = 30
sample_managers_path = DATA_PROCESSED_DIR / 'top_n_sample_managers.csv'
sample_distribution_path = DATA_PROCESSED_DIR / 'top_n_sample_rank_distribution.csv'

standings_candidates_df = pd.read_csv(standings_candidates_path)
standings_candidates_df = standings_candidates_df[
    standings_candidates_df['overall_rank'].astype(int) <= TOP_N
].copy()

sample_managers_df, sample_rank_distribution_df = create_rank_stratified_sample(
    standings_candidates_df,
    sample_size=SAMPLE_SIZE,
    random_seed=RANDOM_SEED,
    my_team_id=MY_TEAM_ID,
    bands=DEFAULT_TOP_80K_RANK_BANDS,
    minimum_per_band=MIN_SAMPLE_MANAGERS_PER_RANK_BAND,
)

sample_managers_df.to_csv(sample_managers_path, index=False)
sample_rank_distribution_df.to_csv(sample_distribution_path, index=False)

expected_sample_size = min(
    SAMPLE_SIZE,
    standings_candidates_df.loc[standings_candidates_df['manager_id'].astype(int) != MY_TEAM_ID, 'manager_id'].nunique(),
)

assert standings_candidates_path.exists()
assert len(sample_managers_df) == expected_sample_size
assert sample_managers_df['manager_id'].is_unique
assert MY_TEAM_ID not in set(sample_managers_df['manager_id'].astype(int))
assert sample_managers_df['rank_band'].notna().all()
assert sample_managers_df['rank_band'].value_counts().min() >= MIN_SAMPLE_MANAGERS_PER_RANK_BAND
assert sample_rank_distribution_df['sample_count'].sum() == len(sample_managers_df)
assert sample_managers_path.exists()
assert sample_distribution_path.exists()

print('Rank-stratified sample created')
print(f'Candidates available: {len(standings_candidates_df):,}')
print(f'Sample managers: {len(sample_managers_df):,}')
print(f'Min rank: {sample_managers_df["overall_rank"].min():,}')
print(f'Median rank: {sample_managers_df["overall_rank"].median():,.0f}')
print(f'Max rank: {sample_managers_df["overall_rank"].max():,}')
print(f'Sample output: {sample_managers_path}')
print(f'Distribution output: {sample_distribution_path}')
print(sample_rank_distribution_df.to_string(index=False))


Rank-stratified sample created
Candidates available: 79,996
Sample managers: 1,000
Min rank: 77
Median rank: 38,056
Max rank: 79,702
Sample output: /Users/djay/Documents/fpl-retrospective-codex-starter/data/processed/top_n_sample_managers.csv
Distribution output: /Users/djay/Documents/fpl-retrospective-codex-starter/data/processed/top_n_sample_rank_distribution.csv
rank_band  candidate_count  sample_count  candidate_share  sample_share
     1-1k             1002            30         0.012526         0.030
    1k-5k             4012            50         0.050153         0.050
   5k-10k             5063            63         0.063291         0.063
  10k-20k             9943           124         0.124294         0.124
  20k-40k            20075           251         0.250950         0.251
  40k-80k            39901           482         0.498787         0.482


## 09 Sample Quality Validation

This section validates whether the rank-stratified sample is large enough across rank bands and saves a reviewer-facing summary table and chart.


In [22]:
import os

os.environ.setdefault('MPLBACKEND', 'Agg')
os.environ.setdefault('MPLCONFIGDIR', str(PROJECT_ROOT / '.matplotlib-cache'))
os.environ.setdefault('XDG_CACHE_HOME', str(PROJECT_ROOT / '.cache'))
Path(os.environ['MPLCONFIGDIR']).mkdir(parents=True, exist_ok=True)
Path(os.environ['XDG_CACHE_HOME']).mkdir(parents=True, exist_ok=True)

import matplotlib.pyplot as plt

sample_managers_path = DATA_PROCESSED_DIR / 'top_n_sample_managers.csv'
sample_quality_summary_path = OUTPUT_TABLES_DIR / 'top_n_sample_summary.csv'
sample_quality_chart_path = OUTPUT_CHARTS_DIR / 'top_n_sample_rank_distribution.png'

sample_managers_df = pd.read_csv(sample_managers_path)
sample_managers_df['overall_rank'] = sample_managers_df['overall_rank'].astype(int)
sample_managers_df['total'] = sample_managers_df['total'].astype(int)
rank_band_order = ['1-1k', '1k-5k', '5k-10k', '10k-20k', '20k-40k', '40k-80k']
sample_managers_df['rank_band'] = pd.Categorical(
    sample_managers_df['rank_band'], categories=rank_band_order, ordered=True
)

required_sample_columns = {'manager_id', 'overall_rank', 'rank_band', 'total'}
missing_sample_columns = required_sample_columns - set(sample_managers_df.columns)

sample_quality_summary_df = (
    sample_managers_df.groupby('rank_band', observed=False)
    .agg(
        sample_count=('manager_id', 'count'),
        min_rank=('overall_rank', 'min'),
        median_rank=('overall_rank', 'median'),
        max_rank=('overall_rank', 'max'),
        min_total_points=('total', 'min'),
        median_total_points=('total', 'median'),
        max_total_points=('total', 'max'),
    )
    .reset_index()
)
sample_quality_summary_df['under_sampled_warning'] = sample_quality_summary_df['sample_count'] < 30
sample_quality_summary_df.to_csv(sample_quality_summary_path, index=False)

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(sample_quality_summary_df['rank_band'].astype(str), sample_quality_summary_df['sample_count'], color='#2f6f73')
ax.axhline(30, color='#b23a48', linestyle='--', linewidth=1, label='30-manager warning threshold')
ax.set_title('Top-N sample managers by rank band')
ax.set_xlabel('Overall rank band')
ax.set_ylabel('Sample managers')
ax.legend(loc='upper left')
ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout()
fig.savefig(sample_quality_chart_path, dpi=150)
plt.close(fig)

assert not missing_sample_columns, f'Missing sample columns: {sorted(missing_sample_columns)}'
assert sample_managers_df['overall_rank'].between(1, TOP_N).all()
assert sample_managers_df['manager_id'].is_unique
assert sample_quality_summary_df['sample_count'].sum() == len(sample_managers_df)
assert sample_quality_summary_path.exists()
assert sample_quality_chart_path.exists()

print('Sample quality validation complete')
print(f'Sample rows: {len(sample_managers_df):,}')
print(f'Min rank: {sample_managers_df["overall_rank"].min():,}')
print(f'Median rank: {sample_managers_df["overall_rank"].median():,.0f}')
print(f'Max rank: {sample_managers_df["overall_rank"].max():,}')
print(f'Summary output: {sample_quality_summary_path}')
print(f'Chart output: {sample_quality_chart_path}')
print(sample_quality_summary_df.to_string(index=False))


Matplotlib is building the font cache; this may take a moment.


Sample quality validation complete
Sample rows: 1,000
Min rank: 77
Median rank: 38,056
Max rank: 79,702
Summary output: /Users/djay/Documents/fpl-retrospective-codex-starter/outputs/tables/top_n_sample_summary.csv
Chart output: /Users/djay/Documents/fpl-retrospective-codex-starter/outputs/charts/top_n_sample_rank_distribution.png
rank_band  sample_count  min_rank  median_rank  max_rank  min_total_points  median_total_points  max_total_points  under_sampled_warning
     1-1k            30        77        596.5       983              2448               2458.0              2492                  False
    1k-5k            50      1139       3089.0      4997              2415               2425.0              2445                  False
   5k-10k            63      5033       7242.0      9946              2398               2406.0              2415                  False
  10k-20k           124     10079      15317.0     19880              2380               2387.0              2398       

## 10 Manager Histories and Transfers

This section fetches and caches manager history and transfer data for manager `816200` and the sampled comparison managers.


In [23]:
from fpl_retro.managers import FAILURE_COLUMNS, build_manager_list, collect_manager_history_and_transfers

RAW_MANAGER_DIR = DATA_RAW_DIR / 'managers'
sample_managers_path = DATA_PROCESSED_DIR / 'top_n_sample_managers.csv'
manager_list_path = DATA_PROCESSED_DIR / 'manager_list.csv'
manager_history_path = DATA_PROCESSED_DIR / 'manager_history.csv'
manager_chips_path = DATA_PROCESSED_DIR / 'manager_chips.csv'
manager_transfers_path = DATA_PROCESSED_DIR / 'manager_transfers.csv'
manager_failures_path = DATA_PROCESSED_DIR / 'manager_fetch_failures.csv'

sample_managers_df = pd.read_csv(sample_managers_path)
manager_list_df = build_manager_list(sample_managers_df, MY_TEAM_ID)
manager_list_df.to_csv(manager_list_path, index=False)

manager_history_df, manager_chips_df, manager_transfers_df, manager_failures_df = collect_manager_history_and_transfers(
    manager_list_df['manager_id'].astype(int).tolist(),
    raw_manager_dir=RAW_MANAGER_DIR,
    force_refresh=not CACHE_ENABLED,
    request_sleep_seconds=REQUEST_SLEEP_SECONDS,
)

manager_history_df.to_csv(manager_history_path, index=False)
manager_chips_df.to_csv(manager_chips_path, index=False)
manager_transfers_df.to_csv(manager_transfers_path, index=False)
manager_failures_df = manager_failures_df.reindex(columns=FAILURE_COLUMNS)
manager_failures_df.to_csv(manager_failures_path, index=False)

assert MY_TEAM_ID in set(manager_list_df['manager_id'].astype(int))
assert MY_TEAM_ID in set(manager_history_df['manager_id'].astype(int))
assert manager_history_df['manager_id'].nunique() >= 1
assert manager_history_df['event'].notna().any()
assert manager_list_path.exists()
assert manager_history_path.exists()
assert manager_chips_path.exists()
assert manager_transfers_path.exists()
assert manager_failures_path.exists()

print('Manager history and transfer collection complete')
print(f'Managers requested: {len(manager_list_df):,}')
print(f'History managers fetched: {manager_history_df["manager_id"].nunique():,}')
print(f'History rows: {len(manager_history_df):,}')
print(f'Chip rows: {len(manager_chips_df):,}')
print(f'Transfer rows: {len(manager_transfers_df):,}')
print(f'Failure rows: {len(manager_failures_df):,}')
print(f'History output: {manager_history_path}')
print(f'Chips output: {manager_chips_path}')
print(f'Transfers output: {manager_transfers_path}')
print(f'Failures output: {manager_failures_path}')


Manager history and transfer collection complete
Managers requested: 1,001
History managers fetched: 1,001
History rows: 38,037
Chip rows: 7,921
Transfer rows: 99,459
Failure rows: 0
History output: /Users/djay/Documents/fpl-retrospective-codex-starter/data/processed/manager_history.csv
Chips output: /Users/djay/Documents/fpl-retrospective-codex-starter/data/processed/manager_chips.csv
Transfers output: /Users/djay/Documents/fpl-retrospective-codex-starter/data/processed/manager_transfers.csv
Failures output: /Users/djay/Documents/fpl-retrospective-codex-starter/data/processed/manager_fetch_failures.csv


## 11 Manager Picks By Gameweek

This section fetches and caches manager picks by gameweek for manager `816200` and a configurable number of sampled comparison managers.


In [26]:
from fpl_retro.managers import PICKS_FAILURE_COLUMNS, collect_manager_picks

RAW_MANAGER_DIR = DATA_RAW_DIR / 'managers'
manager_list_path = DATA_PROCESSED_DIR / 'manager_list.csv'
manager_picks_path = DATA_PROCESSED_DIR / 'manager_picks.csv'
manager_picks_entry_history_path = DATA_PROCESSED_DIR / 'manager_picks_entry_history.csv'
manager_automatic_subs_path = DATA_PROCESSED_DIR / 'manager_automatic_subs.csv'
manager_picks_failures_path = DATA_PROCESSED_DIR / 'manager_picks_failures.csv'

manager_list_df = pd.read_csv(manager_list_path)
focal_manager_ids = manager_list_df.loc[manager_list_df['manager_group'] == 'focal', 'manager_id'].astype(int).tolist()
sample_manager_ids = manager_list_df.loc[manager_list_df['manager_group'] == 'sample', 'manager_id'].astype(int).tolist()
limited_sample_manager_ids = sample_manager_ids[:MAX_MANAGERS_FOR_PICKS]
manager_ids_for_picks = focal_manager_ids + limited_sample_manager_ids

manager_picks_df, manager_picks_entry_history_df, manager_automatic_subs_df, manager_picks_failures_df = collect_manager_picks(
    manager_ids_for_picks,
    SEASON_GWS,
    raw_manager_dir=RAW_MANAGER_DIR,
    force_refresh=not CACHE_ENABLED,
    request_sleep_seconds=REQUEST_SLEEP_SECONDS,
)

players_metadata_path = DATA_PROCESSED_DIR / 'players.csv'
players_metadata_df = pd.read_csv(players_metadata_path)
players_join_columns = [
    column for column in ['id', 'web_name', 'team_name', 'position_name']
    if column in players_metadata_df.columns
]
manager_picks_df = manager_picks_df.merge(
    players_metadata_df[players_join_columns],
    how='left',
    left_on='element',
    right_on='id',
)

manager_picks_df.to_csv(manager_picks_path, index=False)
manager_picks_entry_history_df.to_csv(manager_picks_entry_history_path, index=False)
manager_automatic_subs_df.to_csv(manager_automatic_subs_path, index=False)
manager_picks_failures_df = manager_picks_failures_df.reindex(columns=PICKS_FAILURE_COLUMNS)
manager_picks_failures_df.to_csv(manager_picks_failures_path, index=False)

my_picks = manager_picks_df[manager_picks_df['manager_id'].astype(int) == MY_TEAM_ID]
pick_counts = manager_picks_df.groupby(['manager_id', 'event']).size()

assert MY_TEAM_ID in set(manager_picks_df['manager_id'].astype(int))
assert set(my_picks['event'].astype(int)) == set(SEASON_GWS)
assert pick_counts.mode().iloc[0] == 15
assert {'is_captain', 'is_vice_captain', 'multiplier', 'position'} <= set(manager_picks_df.columns)
assert manager_picks_path.exists()
assert manager_picks_entry_history_path.exists()
assert manager_automatic_subs_path.exists()
assert manager_picks_failures_path.exists()

print('Manager picks collection complete')
print(f'Managers requested: {len(manager_ids_for_picks):,}')
print(f'Manager-gameweeks with picks: {pick_counts.size:,}')
print(f'Pick rows: {len(manager_picks_df):,}')
print(f'Entry-history rows: {len(manager_picks_entry_history_df):,}')
print(f'Automatic substitution rows: {len(manager_automatic_subs_df):,}')
print(f'Failure rows: {len(manager_picks_failures_df):,}')
print(f'Max sampled managers for picks: {MAX_MANAGERS_FOR_PICKS:,}')
print(f'Picks output: {manager_picks_path}')
print(f'Entry-history output: {manager_picks_entry_history_path}')
print(f'Automatic substitutions output: {manager_automatic_subs_path}')
print(f'Failures output: {manager_picks_failures_path}')


Manager picks collection complete
Managers requested: 1,001
Manager-gameweeks with picks: 38,037
Pick rows: 570,555
Entry-history rows: 38,037
Automatic substitution rows: 14,427
Failure rows: 1
Max sampled managers for picks: 1,000
Picks output: /Users/djay/Documents/fpl-retrospective-codex-starter/data/processed/manager_picks.csv
Entry-history output: /Users/djay/Documents/fpl-retrospective-codex-starter/data/processed/manager_picks_entry_history.csv
Automatic substitutions output: /Users/djay/Documents/fpl-retrospective-codex-starter/data/processed/manager_automatic_subs.csv
Failures output: /Users/djay/Documents/fpl-retrospective-codex-starter/data/processed/manager_picks_failures.csv


## 12 Player-Gameweek Features

This section builds leakage-safe player-gameweek features from gameweek live data. Actual gameweek stats remain available as outcomes, while rolling and season-to-date features use only prior gameweeks.


In [ ]:
import numpy as np

from fpl_retro.features import build_player_gw_features


gw_live_path = DATA_PROCESSED_DIR / 'gw_live.csv'
player_gw_features_path = DATA_PROCESSED_DIR / 'player_gw_features.csv'

gw_live_df = pd.read_csv(gw_live_path)
player_gw_features_df = build_player_gw_features(gw_live_df)
player_gw_features_df.to_csv(player_gw_features_path, index=False)

required_feature_columns = {
    'player_id',
    'gameweek',
    'total_points',
    'minutes',
    'total_points_roll3_mean_prior',
    'minutes_roll3_mean_prior',
    'goals_scored_roll3_sum_prior',
    'assists_roll3_sum_prior',
    'expected_goals_roll3_mean_prior',
    'expected_assists_roll3_mean_prior',
    'clean_sheets_roll3_sum_prior',
    'goals_conceded_roll3_sum_prior',
    'saves_roll3_sum_prior',
    'bonus_roll3_sum_prior',
    'bps_roll3_mean_prior',
    'defensive_contribution_roll3_sum_prior',
    'clearances_blocks_interceptions_roll3_sum_prior',
    'recoveries_roll3_sum_prior',
    'tackles_roll3_sum_prior',
    'defensive_contribution_threshold_met_roll3_mean_prior',
    'defensive_contribution_threshold_rate_prior',
    'points_season_to_date_prior',
    'minutes_season_to_date_prior',
    'appearance_rate_prior',
    'start_rate_prior',
    'sixty_plus_rate_prior',
    'points_per_90_prior',
    'xgi_per_90_prior',
    'bps_per_90_prior',
    'defensive_contribution_per_90_prior',
}
missing_feature_columns = required_feature_columns - set(player_gw_features_df.columns)
gw1_features = player_gw_features_df[player_gw_features_df['gameweek'] == 1]
prior_columns = [column for column in player_gw_features_df.columns if column.endswith('_prior')]
rate_columns = [column for column in player_gw_features_df.columns if column.endswith('_rate_prior')]
per_90_columns = [column for column in player_gw_features_df.columns if column.endswith('_per_90_prior')]

assert not missing_feature_columns, f'Missing feature columns: {sorted(missing_feature_columns)}'
assert not player_gw_features_df.empty
assert not player_gw_features_df.duplicated(['player_id', 'gameweek']).any()
assert gw1_features[prior_columns].fillna(0).eq(0).all().all()
assert player_gw_features_df[rate_columns].dropna().ge(0).all().all()
assert player_gw_features_df[rate_columns].dropna().le(1).all().all()
assert np.isfinite(player_gw_features_df[per_90_columns].dropna().to_numpy()).all()
assert player_gw_features_path.exists()

print('Player-gameweek features complete')
print(f'Input rows: {len(gw_live_df):,}')
print(f'Feature rows: {len(player_gw_features_df):,}')
print(f'Players: {player_gw_features_df["player_id"].nunique():,}')
print(f'Gameweeks: {player_gw_features_df["gameweek"].min()}-{player_gw_features_df["gameweek"].max()}')
print(f'Feature columns: {len(player_gw_features_df.columns):,}')
print(f'Output path: {player_gw_features_path}')
print(player_gw_features_df[['player_id', 'gameweek', 'total_points', 'total_points_roll3_mean_prior', 'minutes_roll3_mean_prior', 'defensive_contribution_roll3_sum_prior', 'points_per_90_prior']].head().to_string(index=False))


## 13 Team Strength Features

This section builds leakage-safe team strength features from completed fixtures. Actual gameweek results stay in the table for auditability, while rolling and season-to-date strength columns use only prior gameweeks.


In [ ]:
import numpy as np

from fpl_retro.team_strength import build_team_strength


fixtures_path = DATA_PROCESSED_DIR / 'fixtures.csv'
team_strength_path = DATA_PROCESSED_DIR / 'team_strength.csv'

fixtures_df = pd.read_csv(fixtures_path)
team_strength_df = build_team_strength(fixtures_df)
team_strength_df.to_csv(team_strength_path, index=False)

required_team_strength_columns = {
    'team_id',
    'team_name',
    'team_short_name',
    'gameweek',
    'fixtures_played',
    'goals_for',
    'goals_against',
    'clean_sheets',
    'goals_for_roll3_sum_prior',
    'goals_against_roll3_sum_prior',
    'clean_sheets_roll3_sum_prior',
    'goals_for_per_fixture_prior',
    'goals_against_per_fixture_prior',
    'clean_sheet_rate_prior',
    'home_goals_for_per_home_fixture_prior',
    'away_goals_for_per_away_fixture_prior',
    'xg_like_available',
}
missing_team_strength_columns = required_team_strength_columns - set(team_strength_df.columns)
prior_columns = [column for column in team_strength_df.columns if column.endswith('_prior')]
gw1_strength = team_strength_df[team_strength_df['gameweek'] == 1]
rate_columns = [column for column in team_strength_df.columns if column.endswith('_rate_prior')]
per_fixture_columns = [column for column in team_strength_df.columns if '_per_' in column and column.endswith('_prior')]

assert not missing_team_strength_columns, f'Missing team strength columns: {sorted(missing_team_strength_columns)}'
assert len(team_strength_df) == team_strength_df['team_id'].nunique() * team_strength_df['gameweek'].nunique()
assert not team_strength_df.duplicated(['team_id', 'gameweek']).any()
assert team_strength_df['gameweek'].min() == fixtures_df['event'].min()
assert team_strength_df['gameweek'].max() == fixtures_df['event'].max()
assert gw1_strength[prior_columns].fillna(0).eq(0).all().all()
assert team_strength_df[rate_columns].dropna().ge(0).all().all()
assert team_strength_df[rate_columns].dropna().le(1).all().all()
assert np.isfinite(team_strength_df[per_fixture_columns].dropna().to_numpy()).all()
assert team_strength_path.exists()

print('Team strength features complete')
print(f'Input fixtures: {len(fixtures_df):,}')
print(f'Team strength rows: {len(team_strength_df):,}')
print(f'Teams: {team_strength_df["team_id"].nunique():,}')
print(f'Gameweeks: {team_strength_df["gameweek"].min()}-{team_strength_df["gameweek"].max()}')
print(f'Feature columns: {len(team_strength_df.columns):,}')
print(f'Output path: {team_strength_path}')
print(team_strength_df[['team_id', 'team_short_name', 'gameweek', 'goals_for', 'goals_for_roll3_sum_prior', 'goals_against_roll3_sum_prior', 'clean_sheet_rate_prior']].head().to_string(index=False))


## 14 Fixture Difficulty Features

This section builds fixture outlook features for each team before each gameweek. It combines FPL difficulty ratings with opponent prior team strength, and keeps blank and double gameweek indicators explicit.


In [ ]:
import numpy as np

from fpl_retro.fixture_difficulty import build_fixture_difficulty


fixtures_path = DATA_PROCESSED_DIR / 'fixtures.csv'
team_strength_path = DATA_PROCESSED_DIR / 'team_strength.csv'
fixture_difficulty_path = DATA_PROCESSED_DIR / 'fixture_difficulty.csv'

fixtures_df = pd.read_csv(fixtures_path)
team_strength_df = pd.read_csv(team_strength_path)
fixture_difficulty_df = build_fixture_difficulty(fixtures_df, team_strength_df)
fixture_difficulty_df.to_csv(fixture_difficulty_path, index=False)

required_fixture_difficulty_columns = {
    'team_id',
    'team_name',
    'team_short_name',
    'gameweek',
    'fixture_count_next1',
    'fixture_count_next3',
    'fixture_count_next5',
    'fpl_difficulty_mean_next1',
    'fpl_difficulty_mean_next3',
    'fpl_difficulty_mean_next5',
    'home_fixture_count_next1',
    'away_fixture_count_next1',
    'blank_next1',
    'double_next1',
    'blank_gameweeks_next3',
    'double_gameweeks_next3',
    'opponent_goals_for_per_fixture_prior_mean_next3',
    'opponent_goals_against_per_fixture_prior_mean_next3',
    'opponent_clean_sheet_rate_prior_mean_next5',
}
missing_fixture_difficulty_columns = required_fixture_difficulty_columns - set(fixture_difficulty_df.columns)
rate_columns = [column for column in fixture_difficulty_df.columns if column.endswith('_rate_prior_mean_next3') or column.endswith('_rate_prior_mean_next5')]
numeric_window_columns = [column for column in fixture_difficulty_df.columns if column.startswith(('fixture_count_next', 'blank_next', 'double_next', 'blank_gameweeks_next', 'double_gameweeks_next'))]

assert not missing_fixture_difficulty_columns, f'Missing fixture difficulty columns: {sorted(missing_fixture_difficulty_columns)}'
assert len(fixture_difficulty_df) == fixture_difficulty_df['team_id'].nunique() * fixture_difficulty_df['gameweek'].nunique()
assert not fixture_difficulty_df.duplicated(['team_id', 'gameweek']).any()
assert fixture_difficulty_df['gameweek'].min() == fixtures_df['event'].min()
assert fixture_difficulty_df['gameweek'].max() == fixtures_df['event'].max()
assert fixture_difficulty_df[numeric_window_columns].ge(0).all().all()
assert fixture_difficulty_df[rate_columns].dropna().ge(0).all().all()
assert fixture_difficulty_df[rate_columns].dropna().le(1).all().all()
assert np.isfinite(fixture_difficulty_df.select_dtypes(include='number').dropna().to_numpy()).all()
assert fixture_difficulty_path.exists()

print('Fixture difficulty features complete')
print(f'Input fixtures: {len(fixtures_df):,}')
print(f'Fixture difficulty rows: {len(fixture_difficulty_df):,}')
print(f'Teams: {fixture_difficulty_df["team_id"].nunique():,}')
print(f'Gameweeks: {fixture_difficulty_df["gameweek"].min()}-{fixture_difficulty_df["gameweek"].max()}')
print(f'Feature columns: {len(fixture_difficulty_df.columns):,}')
print(f'Blank team-GWs: {int(fixture_difficulty_df["blank_next1"].sum()):,}')
print(f'Double team-GWs: {int(fixture_difficulty_df["double_next1"].sum()):,}')
print(f'Output path: {fixture_difficulty_path}')
print(fixture_difficulty_df[['team_id', 'team_short_name', 'gameweek', 'fixture_count_next1', 'fpl_difficulty_mean_next1', 'blank_next1', 'double_next1']].head().to_string(index=False))


## 15 My Gameweek Timeline

This section builds one row per gameweek for manager team ID `816200`, combining manager history, chips, transfers, captaincy, bench points, and rank movement.


In [ ]:
from fpl_retro.timeline import build_my_gameweek_timeline


manager_history_path = DATA_PROCESSED_DIR / 'manager_history.csv'
manager_chips_path = DATA_PROCESSED_DIR / 'manager_chips.csv'
manager_transfers_path = DATA_PROCESSED_DIR / 'manager_transfers.csv'
manager_picks_path = DATA_PROCESSED_DIR / 'manager_picks.csv'
player_gw_features_path = DATA_PROCESSED_DIR / 'player_gw_features.csv'
players_path = DATA_PROCESSED_DIR / 'players.csv'
my_timeline_path = DATA_PROCESSED_DIR / 'my_gameweek_timeline.csv'

my_timeline_df = build_my_gameweek_timeline(
    manager_history_df=pd.read_csv(manager_history_path),
    manager_chips_df=pd.read_csv(manager_chips_path),
    manager_transfers_df=pd.read_csv(manager_transfers_path),
    manager_picks_df=pd.read_csv(manager_picks_path),
    player_gw_features_df=pd.read_csv(player_gw_features_path),
    players_df=pd.read_csv(players_path),
    manager_id=MY_TEAM_ID,
)
my_timeline_df.to_csv(my_timeline_path, index=False)

required_timeline_columns = {
    'manager_id',
    'event',
    'points',
    'total_points',
    'overall_rank',
    'overall_rank_movement',
    'rank',
    'gameweek_rank_movement',
    'points_on_bench',
    'bench_raw_points',
    'event_transfers',
    'transfer_count_actual',
    'event_transfers_cost',
    'chip_name',
    'captain_name',
    'captain_base_points',
    'captain_multiplier',
    'captain_points',
    'vice_captain_name',
    'transfer_moves',
}
missing_timeline_columns = required_timeline_columns - set(my_timeline_df.columns)

assert not missing_timeline_columns, f'Missing timeline columns: {sorted(missing_timeline_columns)}'
assert len(my_timeline_df) == len(SEASON_GWS)
assert my_timeline_df['event'].tolist() == SEASON_GWS
assert my_timeline_df['manager_id'].eq(MY_TEAM_ID).all()
assert my_timeline_df['captain_name'].notna().all()
assert my_timeline_df['vice_captain_name'].notna().all()
assert my_timeline_df['pick_count'].eq(15).all()
assert my_timeline_path.exists()

print('My gameweek timeline complete')
print(f'Rows: {len(my_timeline_df):,}')
print(f'Gameweeks: {my_timeline_df["event"].min()}-{my_timeline_df["event"].max()}')
print(f'Columns: {len(my_timeline_df.columns):,}')
print(f'Output path: {my_timeline_path}')
print(my_timeline_df[['event', 'points', 'total_points', 'overall_rank', 'overall_rank_movement', 'chip_name', 'captain_name', 'captain_points', 'points_on_bench', 'transfer_moves']].head().to_string(index=False))


## 16 My Squad Gameweek Reconstruction

This section builds one row per player per gameweek for manager team ID `816200`, adding starter/bench labels, captaincy, actual points, prior player features, prior team strength, and fixture outlook features.


In [ ]:
from fpl_retro.squad import build_my_squad_gameweek


my_squad_path = DATA_PROCESSED_DIR / 'my_squad_gameweek.csv'
my_squad_df = build_my_squad_gameweek(
    manager_picks_df=pd.read_csv(DATA_PROCESSED_DIR / 'manager_picks.csv'),
    player_gw_features_df=pd.read_csv(DATA_PROCESSED_DIR / 'player_gw_features.csv'),
    team_strength_df=pd.read_csv(DATA_PROCESSED_DIR / 'team_strength.csv'),
    fixture_difficulty_df=pd.read_csv(DATA_PROCESSED_DIR / 'fixture_difficulty.csv'),
    teams_df=pd.read_csv(DATA_PROCESSED_DIR / 'teams.csv'),
    manager_id=MY_TEAM_ID,
)
my_squad_df.to_csv(my_squad_path, index=False)

required_squad_columns = {
    'manager_id',
    'event',
    'element',
    'player_web_name',
    'team_id',
    'squad_position',
    'is_starter',
    'is_bench',
    'is_captain',
    'is_vice_captain',
    'multiplier',
    'actual_points',
    'points_after_multiplier',
    'player_points_season_to_date_prior',
    'team_goals_for_per_fixture_prior',
    'fixture_fixture_count_next1',
    'fixture_fpl_difficulty_mean_next3',
}
missing_squad_columns = required_squad_columns - set(my_squad_df.columns)
counts_by_event = my_squad_df.groupby('event').size()
captains_by_event = my_squad_df.groupby('event')['is_captain'].sum()
vice_by_event = my_squad_df.groupby('event')['is_vice_captain'].sum()
starters_by_event = my_squad_df.groupby('event')['is_starter'].sum()
bench_by_event = my_squad_df.groupby('event')['is_bench'].sum()

assert not missing_squad_columns, f'Missing squad columns: {sorted(missing_squad_columns)}'
assert len(my_squad_df) == len(SEASON_GWS) * 15
assert counts_by_event.eq(15).all()
assert captains_by_event.eq(1).all()
assert vice_by_event.eq(1).all()
assert starters_by_event.eq(11).all()
assert bench_by_event.eq(4).all()
assert my_squad_df['actual_points'].notna().all()
assert my_squad_df['team_id'].notna().all()
assert my_squad_path.exists()

print('My squad gameweek reconstruction complete')
print(f'Rows: {len(my_squad_df):,}')
print(f'Gameweeks: {my_squad_df["event"].min()}-{my_squad_df["event"].max()}')
print(f'Columns: {len(my_squad_df.columns):,}')
print(f'Output path: {my_squad_path}')
print(my_squad_df[['event', 'squad_position', 'player_web_name', 'is_starter', 'is_captain', 'is_vice_captain', 'actual_points', 'points_after_multiplier']].head(15).to_string(index=False))


## 17 Transfer Outcome Review

This section evaluates each raw transfer for manager team ID `816200` by comparing transfer-in and transfer-out points from the transfer gameweek over 1 GW, 3 GW, 5 GW, and rest-of-season horizons. Hit cost is allocated evenly across official counted transfers in the gameweek; raw chip-week squad changes receive zero hit allocation.


In [ ]:
from fpl_retro.transfer_outcomes import build_transfer_outcome_review


transfer_outcomes_path = OUTPUT_TABLES_DIR / 'my_transfer_outcomes.csv'
transfer_outcomes_df = build_transfer_outcome_review(
    manager_transfers_df=pd.read_csv(DATA_PROCESSED_DIR / 'manager_transfers.csv'),
    player_gw_features_df=pd.read_csv(DATA_PROCESSED_DIR / 'player_gw_features.csv'),
    players_df=pd.read_csv(DATA_PROCESSED_DIR / 'players.csv'),
    my_timeline_df=pd.read_csv(DATA_PROCESSED_DIR / 'my_gameweek_timeline.csv'),
    manager_id=MY_TEAM_ID,
)
transfer_outcomes_df.to_csv(transfer_outcomes_path, index=False)

required_transfer_outcome_columns = {
    'transfer_sequence',
    'manager_id',
    'event',
    'element_in',
    'in_web_name',
    'element_out',
    'out_web_name',
    'points_in_1gw',
    'points_out_1gw',
    'net_points_1gw',
    'points_in_3gw',
    'points_out_3gw',
    'net_points_3gw',
    'points_in_5gw',
    'points_out_5gw',
    'net_points_5gw',
    'points_in_ros',
    'points_out_ros',
    'net_points_ros',
    'hit_cost_allocated',
    'hit_allocation_assumption',
}
missing_transfer_outcome_columns = required_transfer_outcome_columns - set(transfer_outcomes_df.columns)

assert not missing_transfer_outcome_columns, f'Missing transfer outcome columns: {sorted(missing_transfer_outcome_columns)}'
assert not transfer_outcomes_df.empty
assert transfer_outcomes_df['manager_id'].eq(MY_TEAM_ID).all()
assert transfer_outcomes_df['in_web_name'].notna().all()
assert transfer_outcomes_df['out_web_name'].notna().all()
assert (transfer_outcomes_df['net_points_1gw'] == transfer_outcomes_df['points_in_1gw'] - transfer_outcomes_df['points_out_1gw']).all()
assert (transfer_outcomes_df['net_points_3gw'] == transfer_outcomes_df['points_in_3gw'] - transfer_outcomes_df['points_out_3gw']).all()
assert (transfer_outcomes_df['net_points_5gw'] == transfer_outcomes_df['points_in_5gw'] - transfer_outcomes_df['points_out_5gw']).all()
assert (transfer_outcomes_df['net_points_ros'] == transfer_outcomes_df['points_in_ros'] - transfer_outcomes_df['points_out_ros']).all()
assert transfer_outcomes_path.exists()

print('Transfer outcome review complete')
print(f'Rows: {len(transfer_outcomes_df):,}')
print(f'Gameweeks: {transfer_outcomes_df["event"].min()}-{transfer_outcomes_df["event"].max()}')
print(f'Total raw transfer rows: {len(transfer_outcomes_df):,}')
print(f'Allocated hit cost: {transfer_outcomes_df["hit_cost_allocated"].sum():.1f}')
print(f'Output path: {transfer_outcomes_path}')
print(transfer_outcomes_df[['event', 'in_web_name', 'out_web_name', 'net_points_1gw', 'net_points_3gw', 'net_points_5gw', 'net_points_ros', 'hit_cost_allocated']].head(10).to_string(index=False))


## 18 Transfer Process Features

This section adds leakage-safe context to each transfer from Story 7.1. Player and team process features use prior-only values for the transfer gameweek, while fixture features describe the upcoming schedule from that gameweek.


In [ ]:
from fpl_retro.transfer_outcomes import build_transfer_process_features


transfer_process_path = OUTPUT_TABLES_DIR / 'my_transfer_process_features.csv'
transfer_process_df = build_transfer_process_features(
    transfer_outcomes_df=pd.read_csv(OUTPUT_TABLES_DIR / 'my_transfer_outcomes.csv'),
    player_gw_features_df=pd.read_csv(DATA_PROCESSED_DIR / 'player_gw_features.csv'),
    team_strength_df=pd.read_csv(DATA_PROCESSED_DIR / 'team_strength.csv'),
    fixture_difficulty_df=pd.read_csv(DATA_PROCESSED_DIR / 'fixture_difficulty.csv'),
)
transfer_process_df.to_csv(transfer_process_path, index=False)

required_transfer_process_columns = {
    'transfer_sequence',
    'event',
    'element_in',
    'element_out',
    'in_player_total_points_roll3_mean_prior',
    'out_player_total_points_roll3_mean_prior',
    'diff_player_total_points_roll3_mean_prior',
    'in_player_minutes_roll5_mean_prior',
    'out_player_minutes_roll5_mean_prior',
    'diff_player_minutes_roll5_mean_prior',
    'in_team_points_per_fixture_prior',
    'out_team_points_per_fixture_prior',
    'diff_team_points_per_fixture_prior',
    'in_fixture_fpl_difficulty_mean_next3',
    'out_fixture_fpl_difficulty_mean_next3',
    'diff_fixture_fpl_difficulty_mean_next3',
    'process_feature_note',
}
missing_transfer_process_columns = required_transfer_process_columns - set(transfer_process_df.columns)
process_columns = [column for column in transfer_process_df.columns if column.startswith(('in_player_', 'out_player_', 'diff_player_', 'in_team_', 'out_team_', 'diff_team_', 'in_fixture_', 'out_fixture_', 'diff_fixture_'))]
leaky_process_columns = [column for column in process_columns if any(token in column for token in ['_total_points', '_minutes', '_goals_scored', '_assists', '_clean_sheets']) and not (column.endswith('_prior') or '_prior_' in column)]

assert not missing_transfer_process_columns, f'Missing transfer process columns: {sorted(missing_transfer_process_columns)}'
assert len(transfer_process_df) == len(transfer_outcomes_df)
assert transfer_process_df['transfer_sequence'].is_unique
assert not leaky_process_columns, f'Potential leakage columns: {leaky_process_columns}'
assert transfer_process_df['in_player_total_points_roll3_mean_prior'].notna().any()
assert transfer_process_df['out_player_total_points_roll3_mean_prior'].notna().any()
assert transfer_process_df['in_team_points_per_fixture_prior'].notna().any()
assert transfer_process_df['in_fixture_fpl_difficulty_mean_next3'].notna().any()
assert transfer_process_path.exists()

print('Transfer process features complete')
print(f'Rows: {len(transfer_process_df):,}')
print(f'Columns: {len(transfer_process_df.columns):,}')
print(f'Process columns: {len(process_columns):,}')
print(f'Output path: {transfer_process_path}')
print(transfer_process_df[['event', 'in_web_name', 'out_web_name', 'diff_player_total_points_roll3_mean_prior', 'diff_player_minutes_roll5_mean_prior', 'diff_team_points_per_fixture_prior', 'diff_fixture_fpl_difficulty_mean_next3']].head(10).to_string(index=False))


## 19 Transfer Decision Labels

This section classifies each transfer by separating process quality from outcome quality. It also groups transfers made in the same gameweek so package moves and possible funding legs are evaluated alongside the individual transfer rows.


In [ ]:
from fpl_retro.transfer_outcomes import classify_transfer_decisions, build_transfer_group_decision_labels


transfer_decisions_path = OUTPUT_TABLES_DIR / 'my_transfer_decision_labels.csv'
transfer_group_decisions_path = OUTPUT_TABLES_DIR / 'my_transfer_group_decision_labels.csv'
transfer_decisions_base_df = classify_transfer_decisions(
    pd.read_csv(OUTPUT_TABLES_DIR / 'my_transfer_process_features.csv'),
    decision_quality_labels=DECISION_QUALITY_LABELS,
)
transfer_decisions_df, transfer_group_decisions_df = build_transfer_group_decision_labels(
    transfer_decisions_base_df,
    decision_quality_labels=DECISION_QUALITY_LABELS,
)
transfer_decisions_df.to_csv(transfer_decisions_path, index=False)
transfer_group_decisions_df.to_csv(transfer_group_decisions_path, index=False)

required_transfer_decision_columns = {
    'transfer_sequence',
    'process_score',
    'process_position_route_score',
    'good_process',
    'outcome_score',
    'outcome_correct',
    'decision_quality_key',
    'decision_quality_label',
    'decision_confidence',
    'classification_note',
    'transfer_group_id',
    'group_transfer_count',
    'group_process_score',
    'group_outcome_score',
    'group_decision_quality_key',
    'group_decision_quality_label',
    'possible_funding_leg',
    'individual_vs_group_note',
}
required_transfer_group_columns = {
    'transfer_group_id',
    'manager_id',
    'event',
    'group_transfer_count',
    'group_process_score',
    'group_outcome_score',
    'group_decision_quality_key',
    'group_decision_quality_label',
    'group_net_points_after_hit_5gw',
    'transfer_group_type',
}
missing_transfer_decision_columns = required_transfer_decision_columns - set(transfer_decisions_df.columns)
missing_transfer_group_columns = required_transfer_group_columns - set(transfer_group_decisions_df.columns)
label_counts = transfer_decisions_df['decision_quality_label'].value_counts().rename_axis('decision_quality_label').reset_index(name='transfers')
group_label_counts = transfer_group_decisions_df['group_decision_quality_label'].value_counts().rename_axis('group_decision_quality_label').reset_index(name='transfer_groups')

assert not missing_transfer_decision_columns, f'Missing transfer decision columns: {sorted(missing_transfer_decision_columns)}'
assert not missing_transfer_group_columns, f'Missing transfer group columns: {sorted(missing_transfer_group_columns)}'
assert len(transfer_decisions_df) == len(pd.read_csv(OUTPUT_TABLES_DIR / 'my_transfer_process_features.csv'))
assert transfer_decisions_df['transfer_sequence'].is_unique
assert set(transfer_decisions_df['decision_quality_key']).issubset(set(DECISION_QUALITY_LABELS))
assert set(transfer_decisions_df['group_decision_quality_key']).issubset(set(DECISION_QUALITY_LABELS))
assert set(transfer_group_decisions_df['group_decision_quality_key']).issubset(set(DECISION_QUALITY_LABELS))
assert transfer_decisions_df['decision_quality_label'].isin(DECISION_QUALITY_LABELS.values()).all()
assert transfer_decisions_df['group_decision_quality_label'].isin(DECISION_QUALITY_LABELS.values()).all()
assert transfer_decisions_df['decision_confidence'].isin(['Low', 'Medium', 'High']).all()
assert transfer_decisions_df['process_position_route_score'].between(-3, 3).all()
assert (transfer_decisions_df['outcome_score'] == transfer_decisions_df['net_points_after_hit_5gw']).all()
assert (transfer_group_decisions_df['group_transfer_count'].sum() == len(transfer_decisions_df))
assert (transfer_group_decisions_df['group_net_points_after_hit_5gw'].round(8) == transfer_group_decisions_df['group_outcome_score'].round(8)).all()
assert transfer_decisions_df['possible_funding_leg'].isin([True, False]).all()
assert transfer_decisions_path.exists()
assert transfer_group_decisions_path.exists()

print('Transfer decision labels complete')
print(f'Rows: {len(transfer_decisions_df):,}')
print(f'Transfer groups: {len(transfer_group_decisions_df):,}')
print(f'Output path: {transfer_decisions_path}')
print(f'Group output path: {transfer_group_decisions_path}')
print(label_counts.to_string(index=False))
print(group_label_counts.to_string(index=False))
print(transfer_decisions_df[['event', 'in_web_name', 'out_web_name', 'in_position_short', 'process_position_route_score', 'process_score', 'outcome_score', 'decision_quality_label', 'group_transfer_count', 'group_decision_quality_label', 'possible_funding_leg']].head(12).to_string(index=False))


## 20 Captaincy Review

This section evaluates captaincy choices against the best actual squad option, the best actual starter, and a leakage-safe pre-gameweek candidate score. The candidate score combines prior form, captaincy ceiling, minutes security, position-aware scoring routes, and upcoming fixture context.


In [ ]:
from fpl_retro.evaluation import build_captaincy_review


captaincy_review_path = OUTPUT_TABLES_DIR / 'my_captaincy_review.csv'
captaincy_chart_path = OUTPUT_CHARTS_DIR / 'captaincy_delta.png'

captaincy_review_df = build_captaincy_review(pd.read_csv(DATA_PROCESSED_DIR / 'my_squad_gameweek.csv'))
captaincy_review_df.to_csv(captaincy_review_path, index=False)

fig, ax = plt.subplots(figsize=(12, 5))
bar_colours = ['#2f7d32' if value >= 0 else '#b23b3b' for value in captaincy_review_df['delta_vs_best_starter_extra']]
ax.bar(captaincy_review_df['event'], captaincy_review_df['delta_vs_best_starter_extra'], color=bar_colours)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('Captaincy Delta vs Best Starter')
ax.set_xlabel('Gameweek')
ax.set_ylabel('Extra points gained/lost')
ax.set_xticks(captaincy_review_df['event'])
ax.tick_params(axis='x', labelrotation=90)
fig.tight_layout()
fig.savefig(captaincy_chart_path, dpi=150)
plt.close(fig)

required_captaincy_columns = {
    'event',
    'captain_player_id',
    'captain_name',
    'captain_multiplier',
    'captain_base_points',
    'best_squad_name',
    'best_starter_name',
    'recommended_candidate_name',
    'recommended_candidate_score',
    'captain_candidate_score',
    'captain_ceiling_score',
    'captain_position_route_score',
    'delta_vs_best_squad_extra',
    'delta_vs_best_starter_extra',
    'delta_vs_recommended_candidate_extra',
    'candidate_score_note',
}
missing_captaincy_columns = required_captaincy_columns - set(captaincy_review_df.columns)

assert not missing_captaincy_columns, f'Missing captaincy columns: {sorted(missing_captaincy_columns)}'
assert len(captaincy_review_df) == len(SEASON_GWS)
assert captaincy_review_df['event'].is_unique
assert captaincy_review_df['captain_name'].notna().all()
assert captaincy_review_df['recommended_candidate_name'].notna().all()
assert captaincy_review_df['recommended_candidate_score'].between(0, 1).all()
assert captaincy_review_df['captain_position_route_score'].between(0, 1).all()
assert captaincy_review_path.exists()
assert captaincy_chart_path.exists()

print('Captaincy review complete')
print(f'Rows: {len(captaincy_review_df):,}')
print(f'Output path: {captaincy_review_path}')
print(f'Chart path: {captaincy_chart_path}')
print(f'Total delta vs best starter: {captaincy_review_df["delta_vs_best_starter_extra"].sum():.1f}')
print(f'Total delta vs recommended candidate: {captaincy_review_df["delta_vs_recommended_candidate_extra"].sum():.1f}')
print(captaincy_review_df[['event', 'captain_name', 'captain_base_points', 'best_starter_name', 'best_starter_base_points', 'recommended_candidate_name', 'recommended_candidate_score', 'delta_vs_best_starter_extra', 'delta_vs_recommended_candidate_extra']].head(12).to_string(index=False))


## 21 Benching and Starting XI Review

This section evaluates bench points, high-regret benching, and questionable starting XI decisions. It only treats a benching as questionable when the benched player outscored a formation-valid starter replacement and had at least as strong a pre-gameweek position-aware selection score.


In [ ]:
from fpl_retro.evaluation import build_benching_review, build_benching_gameweek_summary


benching_review_path = OUTPUT_TABLES_DIR / 'my_benching_review.csv'
benching_summary_path = OUTPUT_TABLES_DIR / 'my_benching_gameweek_summary.csv'

benching_review_df = build_benching_review(pd.read_csv(DATA_PROCESSED_DIR / 'my_squad_gameweek.csv'))
benching_summary_df = build_benching_gameweek_summary(benching_review_df)
benching_review_df.to_csv(benching_review_path, index=False)
benching_summary_df.to_csv(benching_summary_path, index=False)

required_benching_columns = {
    'event',
    'bench_player_id',
    'bench_player_name',
    'bench_position',
    'bench_order',
    'bench_actual_points',
    'bench_selection_candidate_score',
    'bench_position_route_score',
    'worst_replaceable_starter_name',
    'bench_regret_points',
    'questionable_benching',
    'high_regret_benching',
    'bench_decision_category',
    'selection_score_note',
}
required_benching_summary_columns = {
    'event',
    'bench_points',
    'bench_regret_points',
    'questionable_benchings',
    'high_regret_benchings',
    'bench_players',
}
missing_benching_columns = required_benching_columns - set(benching_review_df.columns)
missing_benching_summary_columns = required_benching_summary_columns - set(benching_summary_df.columns)

assert not missing_benching_columns, f'Missing benching columns: {sorted(missing_benching_columns)}'
assert not missing_benching_summary_columns, f'Missing benching summary columns: {sorted(missing_benching_summary_columns)}'
assert len(benching_review_df) == len(SEASON_GWS) * 4
assert benching_review_df.groupby('event').size().eq(4).all()
assert benching_summary_df['event'].is_unique
assert benching_summary_df['bench_players'].eq(4).all()
assert benching_review_df['bench_selection_candidate_score'].between(0, 1).all()
assert benching_review_df['bench_position_route_score'].between(0, 1).all()
assert benching_review_df['questionable_benching'].isin([True, False]).all()
assert benching_review_path.exists()
assert benching_summary_path.exists()

print('Benching review complete')
print(f'Rows: {len(benching_review_df):,}')
print(f'Gameweeks: {len(benching_summary_df):,}')
print(f'Total bench points: {benching_summary_df["bench_points"].sum():.1f}')
print(f'Total bench regret points: {benching_summary_df["bench_regret_points"].sum():.1f}')
print(f'Questionable benchings: {int(benching_review_df["questionable_benching"].sum())}')
print(f'Output path: {benching_review_path}')
print(f'Summary path: {benching_summary_path}')
print(benching_review_df.sort_values(['bench_regret_points', 'bench_actual_points'], ascending=False)[['event', 'bench_player_name', 'bench_position', 'bench_actual_points', 'worst_replaceable_starter_name', 'worst_replaceable_starter_points', 'bench_regret_points', 'questionable_benching', 'bench_decision_category']].head(12).to_string(index=False))


## 22 Manager Behaviour Summary

This section creates one season-level behaviour row per manager so manager `816200` can be compared against the top-N sample on points, rank, transfers, hits, captaincy, bench points, chips, value, bank, and common formations. My-team decision review outputs are merged where available so transfer package and position-aware label context is preserved without double-counting it as sample-wide data.


In [ ]:
from fpl_retro.benchmark import build_manager_behaviour_summary


manager_behaviour_summary_path = OUTPUT_TABLES_DIR / 'manager_behaviour_summary.csv'

manager_behaviour_summary_df = build_manager_behaviour_summary(
    manager_history_df=pd.read_csv(DATA_PROCESSED_DIR / 'manager_history.csv'),
    manager_transfers_df=pd.read_csv(DATA_PROCESSED_DIR / 'manager_transfers.csv'),
    manager_chips_df=pd.read_csv(DATA_PROCESSED_DIR / 'manager_chips.csv'),
    manager_picks_df=pd.read_csv(DATA_PROCESSED_DIR / 'manager_picks.csv'),
    top_n_sample_managers_df=pd.read_csv(DATA_PROCESSED_DIR / 'top_n_sample_managers.csv'),
    my_team_id=MY_TEAM_ID,
    my_transfer_decision_labels_df=pd.read_csv(OUTPUT_TABLES_DIR / 'my_transfer_decision_labels.csv'),
    my_transfer_group_decision_labels_df=pd.read_csv(OUTPUT_TABLES_DIR / 'my_transfer_group_decision_labels.csv'),
    my_captaincy_review_df=pd.read_csv(OUTPUT_TABLES_DIR / 'my_captaincy_review.csv'),
    my_benching_gameweek_summary_df=pd.read_csv(OUTPUT_TABLES_DIR / 'my_benching_gameweek_summary.csv'),
)
manager_behaviour_summary_df.to_csv(manager_behaviour_summary_path, index=False)

required_behaviour_columns = {
    'manager_id',
    'is_me',
    'rank_band',
    'final_total_points',
    'final_overall_rank',
    'official_transfers_total',
    'raw_transfer_rows',
    'transfer_hit_cost_total',
    'points_on_bench_total',
    'chip_count',
    'captain_unique_count',
    'most_common_formation',
    'final_squad_value',
    'final_bank',
    'my_transfer_group_count',
    'my_possible_funding_leg_count',
}
missing_behaviour_columns = required_behaviour_columns - set(manager_behaviour_summary_df.columns)
my_behaviour_rows = manager_behaviour_summary_df[manager_behaviour_summary_df['manager_id'].astype(int).eq(MY_TEAM_ID)]
sample_behaviour_rows = manager_behaviour_summary_df[~manager_behaviour_summary_df['manager_id'].astype(int).eq(MY_TEAM_ID)]

assert not missing_behaviour_columns, f'Missing behaviour columns: {sorted(missing_behaviour_columns)}'
assert len(my_behaviour_rows) == 1
assert len(sample_behaviour_rows) > 0
assert manager_behaviour_summary_df['manager_id'].is_unique
assert my_behaviour_rows['is_me'].iloc[0]
assert my_behaviour_rows['my_transfer_group_count'].iloc[0] > 0
assert my_behaviour_rows['my_possible_funding_leg_count'].iloc[0] >= 0
assert manager_behaviour_summary_path.exists()

print('Manager behaviour summary complete')
print(f'Rows: {len(manager_behaviour_summary_df):,}')
print(f'Managers in sample rows: {len(sample_behaviour_rows):,}')
print(f'Output path: {manager_behaviour_summary_path}')
print(manager_behaviour_summary_df.head(12).to_string(index=False))


## 23 Player Adoption Timing

This section identifies key season players and compares the first gameweek manager `816200` owned each player against the sampled managers. Key players are selected by total season points with minimum position coverage so goalkeeper, defender, midfielder, and forward scoring routes remain visible.


In [ ]:
from fpl_retro.benchmark import build_player_adoption_timing


player_adoption_timing_path = OUTPUT_TABLES_DIR / 'player_adoption_timing.csv'

player_adoption_timing_df = build_player_adoption_timing(
    manager_picks_df=pd.read_csv(DATA_PROCESSED_DIR / 'manager_picks.csv'),
    player_gw_features_df=pd.read_csv(DATA_PROCESSED_DIR / 'player_gw_features.csv'),
    top_n_sample_managers_df=pd.read_csv(DATA_PROCESSED_DIR / 'top_n_sample_managers.csv'),
    my_team_id=MY_TEAM_ID,
)
player_adoption_timing_df.to_csv(player_adoption_timing_path, index=False)

required_adoption_columns = {
    'player_id',
    'player_name',
    'position_short',
    'season_points',
    'sample_managers_owned',
    'sample_median_first_owned_gw',
    'my_owned',
    'my_first_owned_gw',
    'my_adoption_delay_vs_sample_median',
    'estimated_points_after_sample_median_before_my_adoption',
    'scoring_route_context',
}
missing_adoption_columns = required_adoption_columns - set(player_adoption_timing_df.columns)

assert not missing_adoption_columns, f'Missing adoption columns: {sorted(missing_adoption_columns)}'
assert player_adoption_timing_df['player_id'].notna().all()
assert player_adoption_timing_df['player_id'].is_unique
assert {'GKP', 'DEF', 'MID', 'FWD'} <= set(player_adoption_timing_df['position_short'])
assert player_adoption_timing_df['sample_median_first_owned_gw'].dropna().between(1, max(SEASON_GWS)).all()
assert player_adoption_timing_df['my_first_owned_gw'].dropna().between(1, max(SEASON_GWS)).all()
assert player_adoption_timing_df['estimated_points_after_sample_median_before_my_adoption'].ge(0).all()
assert player_adoption_timing_path.exists()

print('Player adoption timing review complete')
print(f'Rows: {len(player_adoption_timing_df):,}')
print(f'Output path: {player_adoption_timing_path}')
print(player_adoption_timing_df['position_short'].value_counts().sort_index().to_string())
print(player_adoption_timing_df['adoption_timing_category'].value_counts().to_string())
print(player_adoption_timing_df.head(15).to_string(index=False))


## 24 Squad Structure and Team Exposure Benchmark

This section compares manager `816200` against the sampled managers on squad shape, starter and bench value allocation, premium/mid/budget player mix, exposure to prior strong teams, and exposure to good upcoming fixture runs. Budget fields use the processed player price as a consistent structure proxy because the picks table does not contain each manager's purchase or selling price.


In [ ]:
from fpl_retro.benchmark import build_squad_structure_benchmark


squad_structure_gameweek_path = OUTPUT_TABLES_DIR / 'manager_squad_structure_gameweek.csv'
squad_structure_benchmark_path = OUTPUT_TABLES_DIR / 'squad_structure_benchmark.csv'

squad_structure_gameweek_df, squad_structure_benchmark_df = build_squad_structure_benchmark(
    manager_picks_df=pd.read_csv(DATA_PROCESSED_DIR / 'manager_picks.csv'),
    players_df=pd.read_csv(DATA_PROCESSED_DIR / 'players.csv'),
    team_strength_df=pd.read_csv(DATA_PROCESSED_DIR / 'team_strength.csv'),
    fixture_difficulty_df=pd.read_csv(DATA_PROCESSED_DIR / 'fixture_difficulty.csv'),
    top_n_sample_managers_df=pd.read_csv(DATA_PROCESSED_DIR / 'top_n_sample_managers.csv'),
    my_team_id=MY_TEAM_ID,
    my_transfer_group_decision_labels_df=pd.read_csv(OUTPUT_TABLES_DIR / 'my_transfer_group_decision_labels.csv'),
)
squad_structure_gameweek_df.to_csv(squad_structure_gameweek_path, index=False)
squad_structure_benchmark_df.to_csv(squad_structure_benchmark_path, index=False)

required_structure_columns = {
    'manager_id',
    'is_me',
    'rank_band',
    'gameweeks',
    'avg_squad_value_proxy',
    'avg_bench_value_proxy',
    'avg_bench_value_share_proxy',
    'avg_premium_players',
    'avg_mid_players',
    'avg_budget_players',
    'avg_prior_top6_team_players',
    'avg_good_next3_fixture_players',
    'avg_starter_good_next3_fixture_players',
    'avg_bench_value_proxy_sample_percentile',
    'my_transfer_group_structure_event_count',
    'my_transfer_groups_with_position_value_change_count',
    'value_proxy_note',
}
missing_structure_columns = required_structure_columns - set(squad_structure_benchmark_df.columns)
my_structure_rows = squad_structure_benchmark_df[squad_structure_benchmark_df['manager_id'].astype(int).eq(MY_TEAM_ID)]
sample_structure_rows = squad_structure_benchmark_df[~squad_structure_benchmark_df['manager_id'].astype(int).eq(MY_TEAM_ID)]

assert not missing_structure_columns, f'Missing structure columns: {sorted(missing_structure_columns)}'
assert len(my_structure_rows) == 1
assert len(sample_structure_rows) > 0
assert squad_structure_benchmark_df['manager_id'].is_unique
assert squad_structure_gameweek_df.groupby(['manager_id', 'event']).size().eq(1).all()
assert my_structure_rows['is_me'].iloc[0]
assert my_structure_rows['my_transfer_group_structure_event_count'].iloc[0] > 0
assert my_structure_rows['my_transfer_groups_with_position_value_change_count'].iloc[0] >= 0
assert squad_structure_benchmark_df['avg_bench_value_proxy_sample_percentile'].dropna().between(0, 1).all()
assert squad_structure_gameweek_path.exists()
assert squad_structure_benchmark_path.exists()

print('Squad structure benchmark complete')
print(f'Gameweek rows: {len(squad_structure_gameweek_df):,}')
print(f'Season rows: {len(squad_structure_benchmark_df):,}')
print(f'Gameweek output path: {squad_structure_gameweek_path}')
print(f'Season output path: {squad_structure_benchmark_path}')
print(squad_structure_benchmark_df.head(12).to_string(index=False))


## 25 Transfer Behaviour Benchmark

This section compares transfer behaviour for manager `816200` against the sampled managers. It keeps individual transfer rows separate from manager-gameweek transfer packages so transfer volume and structural package behaviour are not double-counted, and it preserves position-specific incoming-player profiles rather than reducing all transfers to attacking xGI.


In [ ]:
from fpl_retro.benchmark import build_transfer_behaviour_benchmark


manager_transfer_enriched_path = OUTPUT_TABLES_DIR / 'manager_transfer_enriched.csv'
transfer_behaviour_benchmark_path = OUTPUT_TABLES_DIR / 'transfer_behaviour_benchmark.csv'

manager_transfer_enriched_df, transfer_behaviour_benchmark_df = build_transfer_behaviour_benchmark(
    manager_transfers_df=pd.read_csv(DATA_PROCESSED_DIR / 'manager_transfers.csv'),
    manager_history_df=pd.read_csv(DATA_PROCESSED_DIR / 'manager_history.csv'),
    player_gw_features_df=pd.read_csv(DATA_PROCESSED_DIR / 'player_gw_features.csv'),
    fixture_difficulty_df=pd.read_csv(DATA_PROCESSED_DIR / 'fixture_difficulty.csv'),
    top_n_sample_managers_df=pd.read_csv(DATA_PROCESSED_DIR / 'top_n_sample_managers.csv'),
    my_team_id=MY_TEAM_ID,
)
manager_transfer_enriched_df.to_csv(manager_transfer_enriched_path, index=False)
transfer_behaviour_benchmark_df.to_csv(transfer_behaviour_benchmark_path, index=False)

required_transfer_benchmark_columns = {
    'manager_id',
    'is_me',
    'rank_band',
    'transfer_row_count',
    'transfer_package_count',
    'multi_transfer_package_count',
    'hit_transfer_rows',
    'hit_package_count',
    'fixture_improvement_transfer_rate',
    'avg_fixture_difficulty_swing_next3',
    'transfer_in_def_count',
    'transfer_in_mid_count',
    'transfer_in_fwd_count',
    'avg_in_xgi_per_90_prior',
    'avg_in_defensive_contribution_per_90_prior',
    'avg_in_saves_per_90_prior',
    'transfer_row_count_sample_percentile',
    'individual_package_counting_note',
}
missing_transfer_benchmark_columns = required_transfer_benchmark_columns - set(transfer_behaviour_benchmark_df.columns)
my_transfer_benchmark_rows = transfer_behaviour_benchmark_df[transfer_behaviour_benchmark_df['manager_id'].astype(int).eq(MY_TEAM_ID)]
sample_transfer_benchmark_rows = transfer_behaviour_benchmark_df[~transfer_behaviour_benchmark_df['manager_id'].astype(int).eq(MY_TEAM_ID)]

assert not missing_transfer_benchmark_columns, f'Missing transfer benchmark columns: {sorted(missing_transfer_benchmark_columns)}'
assert len(my_transfer_benchmark_rows) == 1
assert len(sample_transfer_benchmark_rows) > 0
assert transfer_behaviour_benchmark_df['manager_id'].is_unique
assert manager_transfer_enriched_df['transfer_group_id'].notna().all()
assert my_transfer_benchmark_rows['transfer_row_count'].iloc[0] >= my_transfer_benchmark_rows['transfer_package_count'].iloc[0]
assert my_transfer_benchmark_rows['transfer_row_count'].iloc[0] > 0
assert transfer_behaviour_benchmark_df['transfer_row_count_sample_percentile'].dropna().between(0, 1).all()
assert transfer_behaviour_benchmark_df['fixture_improvement_transfer_rate_sample_percentile'].dropna().between(0, 1).all()
assert manager_transfer_enriched_path.exists()
assert transfer_behaviour_benchmark_path.exists()

print('Transfer behaviour benchmark complete')
print(f'Enriched transfer rows: {len(manager_transfer_enriched_df):,}')
print(f'Benchmark rows: {len(transfer_behaviour_benchmark_df):,}')
print(f'Enriched output path: {manager_transfer_enriched_path}')
print(f'Benchmark output path: {transfer_behaviour_benchmark_path}')
print(transfer_behaviour_benchmark_df.head(12).to_string(index=False))


## 26 Candidate Rule Features

This section builds the leakage-safe feature base for rule testing. Feature columns are prefixed with `feature_` and use only prior player form, team strength, fixture outlook, price, position, and position-aware scoring routes. Future 1GW, 3GW, and 5GW point windows are prefixed with `outcome_` so later rule tests can evaluate decisions without mixing outcomes into pre-gameweek inputs.


In [ ]:
from fpl_retro.rules import build_candidate_rule_features, validate_candidate_rule_features


candidate_rule_features_path = OUTPUT_TABLES_DIR / 'candidate_rule_features.csv'

candidate_rule_features_df = build_candidate_rule_features(
    player_gw_features_df=pd.read_csv(DATA_PROCESSED_DIR / 'player_gw_features.csv'),
    team_strength_df=pd.read_csv(DATA_PROCESSED_DIR / 'team_strength.csv'),
    fixture_difficulty_df=pd.read_csv(DATA_PROCESSED_DIR / 'fixture_difficulty.csv'),
    manager_picks_df=pd.read_csv(DATA_PROCESSED_DIR / 'manager_picks.csv'),
)
candidate_rule_features_df.to_csv(candidate_rule_features_path, index=False)

rule_feature_checks = validate_candidate_rule_features(candidate_rule_features_df)
assert rule_feature_checks['rows'] > 0
assert rule_feature_checks['duplicated_player_gameweeks'] == 0
assert not rule_feature_checks['missing_route_columns'], rule_feature_checks['missing_route_columns']
assert not rule_feature_checks['missing_outcome_columns'], rule_feature_checks['missing_outcome_columns']
assert not rule_feature_checks['leaked_feature_columns'], rule_feature_checks['leaked_feature_columns']
assert candidate_rule_features_path.exists()

rule_feature_checks


## 27 Player Selection Rule Candidates

This section tests transparent player-selection rules against future point outcomes. Rules are evaluated separately by position and compared with same-position, same-gameweek baselines, so goalkeeper, defender, midfielder, and forward scoring routes are not pooled into a misleading average.


In [ ]:
from fpl_retro.rules import build_player_selection_rule_results, validate_player_selection_rule_results


player_selection_rule_candidates_path = OUTPUT_TABLES_DIR / 'player_selection_rule_candidates.csv'

player_selection_rule_candidates_df = build_player_selection_rule_results(
    pd.read_csv(OUTPUT_TABLES_DIR / 'candidate_rule_features.csv')
)
player_selection_rule_candidates_df.to_csv(player_selection_rule_candidates_path, index=False)

player_selection_rule_checks = validate_player_selection_rule_results(player_selection_rule_candidates_df)
assert player_selection_rule_checks['rows'] > 0
assert player_selection_rule_checks['rules'] >= 8
assert set(player_selection_rule_checks['positions']) == {'DEF', 'FWD', 'GKP', 'MID'}
assert not player_selection_rule_checks['missing_columns'], player_selection_rule_checks['missing_columns']
assert player_selection_rule_checks['duplicate_rule_position_rows'] == 0
assert player_selection_rule_checks['min_sample_size'] >= 30
assert player_selection_rule_candidates_path.exists()

player_selection_rule_checks


## 28 Transfer Rule Candidates

This section tests transfer-style rules for both individual transfer legs and manager-gameweek transfer packages. Individual rows compare transferred-in and transferred-out future points with hit cost allocated across counted transfers, while package rows aggregate each manager-gameweek once so structural transfer packages are not double-counted as independent strategic decisions.


In [ ]:
from fpl_retro.rules import build_transfer_rule_results, validate_transfer_rule_results


transfer_rule_enriched_path = OUTPUT_TABLES_DIR / 'transfer_rule_enriched.csv'
transfer_rule_packages_path = OUTPUT_TABLES_DIR / 'transfer_rule_packages.csv'
transfer_rule_candidates_path = OUTPUT_TABLES_DIR / 'transfer_rule_candidates.csv'

transfer_rule_enriched_df, transfer_rule_packages_df, transfer_rule_candidates_df = build_transfer_rule_results(
    manager_transfer_enriched_df=pd.read_csv(OUTPUT_TABLES_DIR / 'manager_transfer_enriched.csv'),
    candidate_rule_features_df=pd.read_csv(OUTPUT_TABLES_DIR / 'candidate_rule_features.csv'),
)
transfer_rule_enriched_df.to_csv(transfer_rule_enriched_path, index=False)
transfer_rule_packages_df.to_csv(transfer_rule_packages_path, index=False)
transfer_rule_candidates_df.to_csv(transfer_rule_candidates_path, index=False)

transfer_rule_checks = validate_transfer_rule_results(transfer_rule_candidates_df, transfer_rule_packages_df)
assert transfer_rule_checks['rows'] > 0
assert {'individual_transfer_leg', 'transfer_package'} <= set(transfer_rule_checks['rule_levels'])
assert not transfer_rule_checks['missing_columns'], transfer_rule_checks['missing_columns']
assert transfer_rule_checks['duplicate_rule_rows'] == 0
assert transfer_rule_checks['duplicate_package_ids'] == 0
assert transfer_rule_checks['min_sample_size'] >= 30
assert transfer_rule_candidates_path.exists()
assert transfer_rule_packages_path.exists()

transfer_rule_checks


## 29 Next-Season Rulebook

This section converts player-selection and transfer-rule evidence into plain-English next-season rules. It keeps player selection, individual transfer-leg, and transfer-package guidance separate, includes confidence and overfitting caveats, and links transfer package rules back to my season where matching examples exist.


In [ ]:
from fpl_retro.rules import build_next_season_rulebook, validate_next_season_rulebook


next_season_rulebook_path = OUTPUT_TABLES_DIR / 'next_season_rulebook.csv'

next_season_rulebook_df = build_next_season_rulebook(
    player_rule_results_df=pd.read_csv(OUTPUT_TABLES_DIR / 'player_selection_rule_candidates.csv'),
    transfer_rule_results_df=pd.read_csv(OUTPUT_TABLES_DIR / 'transfer_rule_candidates.csv'),
    transfer_rule_enriched_df=pd.read_csv(OUTPUT_TABLES_DIR / 'transfer_rule_enriched.csv'),
    transfer_rule_packages_df=pd.read_csv(OUTPUT_TABLES_DIR / 'transfer_rule_packages.csv'),
    my_transfer_group_decision_labels_df=pd.read_csv(OUTPUT_TABLES_DIR / 'my_transfer_group_decision_labels.csv'),
    adoption_timing_df=pd.read_csv(OUTPUT_TABLES_DIR / 'player_adoption_timing.csv'),
    captaincy_review_df=pd.read_csv(OUTPUT_TABLES_DIR / 'my_captaincy_review.csv'),
    benching_summary_df=pd.read_csv(OUTPUT_TABLES_DIR / 'my_benching_gameweek_summary.csv'),
)
next_season_rulebook_df.to_csv(next_season_rulebook_path, index=False)

next_season_rulebook_checks = validate_next_season_rulebook(next_season_rulebook_df)
assert next_season_rulebook_checks['rows'] > 0
assert not next_season_rulebook_checks['missing_columns'], next_season_rulebook_checks['missing_columns']
assert next_season_rulebook_checks['duplicate_rule_ids'] == 0
assert {'player_selection', 'individual_transfer_leg', 'transfer_package'} <= set(next_season_rulebook_checks['rule_scopes'])
assert next_season_rulebook_checks['low_confidence_labelled']
assert next_season_rulebook_checks['empty_rule_text_rows'] == 0
assert next_season_rulebook_path.exists()

next_season_rulebook_checks


## 30 Season Gap and Leak Summary

This section combines the decision review outputs into a ranked season leak and gain summary. Transfer strategy is scored at package level to avoid double-counting individual legs, while individual transfer rows remain diagnostic evidence for within-package review. Additive rows feed the waterfall chart; diagnostic rows provide caveats where the current data does not support a direct points estimate.


In [ ]:
from fpl_retro.final_report import (
    build_season_gap_leak_summary,
    plot_season_gap_waterfall,
    validate_season_gap_leak_summary,
)


season_gap_leak_summary_path = OUTPUT_TABLES_DIR / 'season_gap_leak_summary.csv'
season_gap_waterfall_path = OUTPUT_CHARTS_DIR / 'season_gap_waterfall.png'

season_gap_leak_summary_df = build_season_gap_leak_summary(
    transfer_groups_df=pd.read_csv(OUTPUT_TABLES_DIR / 'my_transfer_group_decision_labels.csv'),
    transfer_decisions_df=pd.read_csv(OUTPUT_TABLES_DIR / 'my_transfer_decision_labels.csv'),
    captaincy_review_df=pd.read_csv(OUTPUT_TABLES_DIR / 'my_captaincy_review.csv'),
    benching_summary_df=pd.read_csv(OUTPUT_TABLES_DIR / 'my_benching_gameweek_summary.csv'),
    adoption_timing_df=pd.read_csv(OUTPUT_TABLES_DIR / 'player_adoption_timing.csv'),
    squad_structure_benchmark_df=pd.read_csv(OUTPUT_TABLES_DIR / 'squad_structure_benchmark.csv'),
    manager_behaviour_summary_df=pd.read_csv(OUTPUT_TABLES_DIR / 'manager_behaviour_summary.csv'),
)
season_gap_leak_summary_df.to_csv(season_gap_leak_summary_path, index=False)
plot_season_gap_waterfall(season_gap_leak_summary_df, season_gap_waterfall_path)

season_gap_leak_summary_checks = validate_season_gap_leak_summary(
    season_gap_leak_summary_df,
    season_gap_waterfall_path,
)
assert season_gap_leak_summary_checks['rows'] > 0
assert not season_gap_leak_summary_checks['missing_columns'], season_gap_leak_summary_checks['missing_columns']
assert season_gap_leak_summary_checks['evidence_missing_rows'] == 0
assert season_gap_leak_summary_checks['has_package_basis']
assert season_gap_leak_summary_checks['has_individual_diagnostic']
assert {'transfer_package', 'captaincy', 'benching', 'player_adoption', 'squad_structure', 'chips'} <= set(season_gap_leak_summary_checks['decision_types'])
assert season_gap_leak_summary_checks['chart_exists']
assert season_gap_leak_summary_path.exists()

season_gap_leak_summary_checks


## 31 Final Executive Summary

This retrospective analyses team `816200` against the sampled top-N managers and converts the decision review into next-season rules. The key supporting outputs are `outputs/tables/season_gap_leak_summary.csv`, `outputs/charts/season_gap_waterfall.png`, `outputs/tables/next_season_rulebook.csv`, `outputs/tables/my_transfer_group_decision_labels.csv`, `outputs/tables/my_transfer_decision_labels.csv`, `outputs/tables/my_captaincy_review.csv`, `outputs/tables/my_benching_gameweek_summary.csv`, `outputs/tables/player_adoption_timing.csv`, and `outputs/tables/squad_structure_benchmark.csv`.

### Headline findings

The biggest apparent leak was slow adoption of key players. The leak table estimates about 1,040 points after the sampled managers' median adoption point across 18 late or never-owned key players. This is deliberately labelled Low confidence because those players are selected with hindsight, but the examples are still useful: João Pedro was first owned in Gameweek 30 versus the sample median of Gameweek 1 and had 160 estimated missed points; Semenyo was first owned in Gameweek 23 versus the sample median of Gameweek 4 and had 99 estimated missed points; Donnarumma was never owned and had 103 estimated missed points after the sample median. This is not a claim that all those points were realistically recoverable, but it strongly suggests that high-adoption players with secure minutes and position-specific scoring routes need faster review.

The clearest medium-confidence leaks were weekly team-selection decisions. Benching and starting XI choices produced 217 formation-valid regret points, with 40 questionable benchings and 18 high-regret benchings. Captaincy was 178 points behind the best actual starter, but that is a hindsight ceiling rather than a realistic target. Against the pre-gameweek candidate benchmark, captaincy was 94 points positive, so the lesson is not to chase perfect hindsight; it is to avoid obvious misses while keeping a process-led captaincy shortlist.

Transfers were a net positive at package level. Across 28 transfer groups, the package-level 5-gameweek net after hits was +350 points. This is the number to use for strategic transfer conclusions because same-gameweek transfers can be linked: one downgrade may fund an upgrade elsewhere. The individual transfer legs are therefore diagnostic only. The table flags 38 negative individual legs and 12 possible funding legs, but these are not added to the headline transfer total. The true review question is whether the whole package improved the squad.

The bad transfer evidence is still useful. Negative packages totalled -116 points, and official hit cost was 36 points, but both are diagnostic rows rather than extra additive leaks. The worst package was Gameweek 22, a two-transfer spent-budget package that lost 32 points after a 4-point hit. Other negative packages included Gameweek 27, Gameweek 12, and Gameweek 33. These are the weeks to inspect for whether the package had enough minutes, fixture, role, price, and position-route justification before the move.

### Position-specific lessons

Goalkeepers should not be evaluated like attackers. The supported goalkeeper rules favoured secure minutes, strong position-relevant route score, and goalkeeper save route. Donnarumma and Kelleher also show why clean sheets, saves, and bonus can create meaningful value even with little attacking involvement.

Defenders need clean-sheet, defensive-contribution, bonus, and attacking context together. The adoption table highlights Virgil and Matheus N. as defender examples where clean sheets and defensive contributions mattered materially. Transfer rules also support clean-sheet-route upgrades for defenders, but fixture-only defender claims should be treated carefully when the player lacks minutes or route strength.

Midfielders should not be reduced to xGI. Strong xGI still matters, but midfield value also came through secure minutes, premium secure route, team strength, and defensive contributions. Semenyo, Garner, B.Fernandes, and Szoboszlai are examples where defensive contribution points were part of the scoring route, not noise.

Forwards remain the most attacking-route dependent position. The strongest forward rules were secure minutes, strong xGI proxy, strong recent points, and high attacking route score. A forward transfer should improve expected goal involvement or minutes security; a fixture improvement alone is not enough.

### Next-season operating system

1. Review the top-sample adoption table every gameweek for players with high sample adoption, secure minutes, and a position-specific route to points. Treat this as an alert, not an automatic buy list.
2. Make transfers as packages. Write down the package purpose, the player sacrificed, the funded upgrade, the hit cost, and the 3-5 gameweek route to points before confirming.
3. Do not score transfer legs twice. Use package-level outcomes for strategy and individual-leg rows only to explain what happened inside the package.
4. Captain from a short pre-gameweek candidate list built on minutes security, role, route score, and fixtures. Review misses against the candidate list, not only against the best actual scorer.
5. Start players using position-aware scoring routes: saves for goalkeepers, clean sheets and defensive contributions for defenders, mixed attacking and defensive routes for midfielders, and xGI plus minutes for forwards.
6. Treat Bench Boost and bench depth as a planning problem. Benching regret was large enough that the weekly XI decision deserves the same discipline as transfers and captaincy.
7. Use confidence labels. High-confidence rules can shape defaults; Medium-confidence rules need squad-context checks; Low-confidence findings are prompts for review, not instructions.

### Uncertainty

The final report is retrospective. It uses leakage-safe features where rules are tested, but some summary estimates are still hindsight summaries. Late adoption, best-starter captaincy, and bench regret can overstate recoverable points. Transfer package outcomes are more robust for strategic transfer review because they respect bundled decisions and hit costs, but even those are affected by injuries, rotation, price changes, and luck. The next-season value is in improving the repeatable process, not in assuming the exact point totals would recur.


## 11 Weekly Decision System

This section starts the next phase: a reusable weekly decision pack for transfer candidates, sell candidates, transfer packages, captaincy, and a plain-English weekly summary. Story 11.1 only verifies the orchestration skeleton; scoring logic is added in later stories.


In [ ]:
from fpl_retro.weekly_decision_system import (
    build_learned_weekly_decision_rules,
    build_transfer_package_review,
    build_transfer_pair_review,
    build_weekly_decision_pack,
    save_weekly_hit_payoff_curve,
)


weekly_pack = build_weekly_decision_pack(
    target_gw=10,
    manager_id=MY_TEAM_ID,
    free_transfers=1,
    bank=0.0,
    candidate_rule_features=OUTPUT_TABLES_DIR / 'candidate_rule_features.csv',
    current_squad=DATA_PROCESSED_DIR / 'my_squad_gameweek.csv',
    output_tables_dir=OUTPUT_TABLES_DIR,
)

expected_weekly_pack_keys = {
    'current_squad_context',
    'transfer_candidate_shortlist',
    'sell_candidate_review',
    'transfer_package_review',
    'captaincy_decision',
    'weekly_summary',
}
assert set(weekly_pack) == expected_weekly_pack_keys
assert all(not table.empty for table in weekly_pack.values())
assert len(weekly_pack['current_squad_context']) == 15
assert weekly_pack['current_squad_context']['element'].is_unique
assert weekly_pack['current_squad_context']['is_captain'].sum() == 1
assert weekly_pack['current_squad_context']['is_vice_captain'].sum() == 1

transfer_candidate_shortlist_df = weekly_pack['transfer_candidate_shortlist']
owned_player_ids = set(weekly_pack['current_squad_context']['element'])
assert not set(transfer_candidate_shortlist_df['player_id']).intersection(owned_player_ids)
assert transfer_candidate_shortlist_df['position_short'].nunique() >= 3
score_columns = [
    'role_security_score',
    'route_to_points_score',
    'fixture_score',
    'team_strength_score',
    'price_value_score',
    'ownership_or_adoption_score',
    'transfer_candidate_score',
]
assert transfer_candidate_shortlist_df[score_columns].notna().all().all()
assert transfer_candidate_shortlist_df[score_columns].apply(lambda col: col.between(0, 1).all()).all()
assert (OUTPUT_TABLES_DIR / 'weekly_transfer_candidate_shortlist.csv').exists()

sell_candidate_review_df = weekly_pack['sell_candidate_review']
assert len(sell_candidate_review_df) == 15
assert set(sell_candidate_review_df['element']) == owned_player_ids
assert sell_candidate_review_df['sell_recommendation'].isin(['sell', 'monitor', 'hold']).all()
assert sell_candidate_review_df['confidence'].isin(['High', 'Medium', 'Low']).all()
assert sell_candidate_review_df['sell_explanation'].notna().all()
assert sell_candidate_review_df['hold_explanation'].notna().all()
sell_score_columns = [
    'role_security_risk_score',
    'position_route_risk_score',
    'fixture_risk_score',
    'team_context_risk_score',
    'price_value_risk_score',
    'squad_role_risk_score',
    'opportunity_cost_score',
    'sell_risk_score',
]
assert sell_candidate_review_df[sell_score_columns].notna().all().all()
assert sell_candidate_review_df[sell_score_columns].apply(lambda col: col.between(0, 1).all()).all()
leakage_columns = {'actual_points', 'points_after_multiplier', 'player_total_points'}
assert not leakage_columns.intersection(sell_candidate_review_df.columns)
assert (OUTPUT_TABLES_DIR / 'weekly_sell_candidate_review.csv').exists()

learned_weekly_rules = build_learned_weekly_decision_rules(
    candidate_rule_features=OUTPUT_TABLES_DIR / 'candidate_rule_features.csv',
    player_selection_rule_results=OUTPUT_TABLES_DIR / 'player_selection_rule_candidates.csv',
    manager_transfers=DATA_PROCESSED_DIR / 'manager_transfers.csv',
    manager_picks=DATA_PROCESSED_DIR / 'manager_picks.csv',
    top_n_sample_managers=DATA_PROCESSED_DIR / 'top_n_sample_managers.csv',
    output_tables_dir=OUTPUT_TABLES_DIR,
)
required_learned_rule_columns = {
    'rule_id',
    'decision_area',
    'plain_english_rule',
    'sample_size',
    'sample_manager_count',
    'rank_band_coverage',
    'evidence_window',
    'mean_outcome',
    'median_outcome',
    'baseline_outcome',
    'uplift_vs_baseline',
    'confidence',
    'when_to_use',
    'when_to_ignore',
    'risk_of_overfitting',
}
for learned_rules_df in learned_weekly_rules.values():
    assert not learned_rules_df.empty
    assert required_learned_rule_columns.issubset(learned_rules_df.columns)
    assert learned_rules_df['evidence_window'].isin(['1GW', '3GW', '5GW']).all()
    assert learned_rules_df['sample_manager_count'].min() > 0
    assert learned_rules_df['confidence'].isin(['High', 'Medium', 'Low']).all()
assert (OUTPUT_TABLES_DIR / 'learned_candidate_shortlist_rules.csv').exists()
assert (OUTPUT_TABLES_DIR / 'learned_sell_hold_rules.csv').exists()

transfer_pair_review_df = build_transfer_pair_review(
    transfer_candidate_shortlist_df,
    sell_candidate_review_df,
    bank=0.0,
    learned_candidate_rules_df=learned_weekly_rules['learned_candidate_shortlist_rules'],
    learned_sell_hold_rules_df=learned_weekly_rules['learned_sell_hold_rules'],
)
transfer_pair_review_df.to_csv(OUTPUT_TABLES_DIR / 'weekly_transfer_pair_review.csv', index=False)
assert not transfer_pair_review_df.empty
assert transfer_pair_review_df['sell_position'].eq(transfer_pair_review_df['buy_position']).all()
assert transfer_pair_review_df['is_affordable'].isin([True, False]).all()
assert transfer_pair_review_df.loc[
    transfer_pair_review_df['recommendation'].ne('unaffordable'),
    'is_affordable',
].all()
pair_score_columns = [
    'fixture_swing_score',
    'route_swing_score',
    'role_security_swing_score',
    'team_strength_swing_score',
    'value_swing_score',
    'learned_evidence_score',
    'upgrade_score',
]
assert transfer_pair_review_df[pair_score_columns].notna().all().all()
assert transfer_pair_review_df[pair_score_columns].apply(lambda col: col.between(0, 1).all()).all()
assert (OUTPUT_TABLES_DIR / 'weekly_transfer_pair_review.csv').exists()

transfer_package_review_df = build_transfer_package_review(
    transfer_pair_review_df,
    free_transfers=1,
    transfer_rule_candidates_df=pd.read_csv(OUTPUT_TABLES_DIR / 'transfer_rule_candidates.csv'),
)
transfer_package_review_df.to_csv(OUTPUT_TABLES_DIR / 'weekly_transfer_package_review.csv', index=False)
save_weekly_hit_payoff_curve(
    transfer_package_review_df,
    OUTPUT_CHARTS_DIR / 'weekly_hit_payoff_curve.png',
)
assert not transfer_package_review_df.empty
assert set(['0', '-4', '-8', '-12']).issubset(set(transfer_package_review_df['hit_scenario']))
assert transfer_package_review_df['duplicate_check_passed'].all()
assert transfer_package_review_df['hit_cost'].ge(0).all()
assert transfer_package_review_df['recommendation'].isin([
    'roll_transfer',
    'hold',
    'use_free_transfer',
    'consider_minus_4',
    'consider_minus_8',
    'consider_minus_12',
    'avoid_hit',
]).all()
assert (OUTPUT_TABLES_DIR / 'weekly_transfer_package_review.csv').exists()
assert (OUTPUT_CHARTS_DIR / 'weekly_hit_payoff_curve.png').exists()

{key: table.shape for key, table in weekly_pack.items()}
